# Supplementary Code S1, notebook 01: main analysis

Runs the cross-fitted MIMIC-CXR analysis, the additional MIMIC cohort, the NIH ChestX-ray14 external comparison, the NegBio label check, the MIMIC subgroup audit and the queue simulation reported in the article. Designed for Google Colab with Google Drive storage on a CPU runtime.

**Runtime.** A full run takes about 4–6 hours on a free 2-core Colab runtime (three fold assignments, the ensemble signal and encoding of the NIH images). For a faster run, set `seeds=[0]`, `gate_train='dev'` and `ensemble_signal=False` in the constants cell, and set `os.environ['ARG_N_ZIPS']='2'` before section 8b. Intermediate results are checkpointed. Only section 8b downloads data (public NIH image archives of about 2 GB each, and the BiomedCLIP weights).

**Inputs.** The private cached MIMIC embeddings, manifests and configuration files described in README.txt. For section 7b, place `mimic-cxr-2.0.0-negbio.csv.gz` next to the CheXpert label file on Drive or set `NEGBIO_PATH`. Section 6e runs when MIMIC-IV `patients.csv.gz` (and optionally `admissions.csv.gz`) is placed in `<run folder>/mimic_iv/` or `MIMIC_IV_DIR` is set.

**Scope.** Request costs, reviewer accuracy and queue behavior are modeled assumptions examined with sensitivity grids. The NIH comparison is image-only and uses eight shared labels. Subgroup audits are descriptive.

In [ ]:
from pathlib import Path
import sys, json, hashlib, pickle, time, os, platform, re, warnings, gc
import importlib.util, subprocess
DRIVE_MOUNT = '/content/drive_fresh'
RUN_NAME = 'full_20261015_171819_876684'
RUN_DIR = Path(DRIVE_MOUNT)/'MyDrive/adaptive_risk_gate_clean/runs'/RUN_NAME
V13_REL = 'methodology_v13_final/d8228aba22e5bffd9bc8'     # fresh 2,000-patient cohort (features already cached)
packages = {'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','scipy':'scipy',
            'sklearn':'scikit-learn','openpyxl':'openpyxl','threadpoolctl':'threadpoolctl'}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore')
print('Same source run:', RUN_DIR)
print('Mode: cached embeddings only. No download, no image/report extraction, no encoder execution.')


In [ ]:
# Reuse an active mount; mount once if needed; never force-remount.
if not RUN_DIR.is_dir():
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Open this notebook in Colab with the same Drive account as the earlier runs.') from exc
    mount = Path(DRIVE_MOUNT)
    if not os.path.ismount(mount):
        if mount.exists() and any(mount.iterdir()):
            raise RuntimeError(f'{mount} is nonempty but not an active Drive mount. Resolve it; nothing was changed.')
        drive.mount(DRIVE_MOUNT)
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f'Completed run not found: {RUN_DIR}')
required = ['persistent_encoded_data_PRIVATE.pkl', 'frozen_manifest.json', 'config.json',
            'encoder_manifest.json', 'cohort_summary.csv']
absent = [n for n in required if not (RUN_DIR/n).is_file()]
if absent:
    raise FileNotFoundError('Saved inputs missing: ' + ', '.join(absent) + '. Nothing is re-extracted.')
print('All required cached inputs present.')


## 0. Fixed constants, helpers and analysis settings

In [ ]:
from scipy.special import expit, logit, xlogy
from scipy.stats import binom
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, average_precision_score

LABELS = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion',
          'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
          'Pleural Other', 'Fracture', 'Support Devices']
SEVERITY = np.array([0, .45, .45, .5, .65, .8, .7, .8, .4, 1, .6, .35, .5, .3])
SEVERE = SEVERITY >= .7
PARTS = ['train', 'tune', 'gate_fit', 'gate_select', 'calibration', 'test']
EPS = 1e-6
CUTOFF = 0.5

# ------------------------------------------------------------------ fixed analysis plan (written BEFORE any result)
PLAN = dict(
    method='ARG_V19_NEGBIO_SUBGROUPS_NIH',
    folds=6,                       # patient-grouped outer folds: fit(K-3) / dev / cal / eval
    seed=20261019,
    fp_weight=1.0,                 # same development assumption as V4-V6
    conformal_alpha=0.10,
    delta=0.10,                    # LTT failure probability
    harm_kinds={'fn': [0.01, 0.02, 0.03, 0.04],            # severity-weighted missed-finding burden per case
                'severe': [0.05, 0.10, 0.15, 0.20]},        # critical-finding missed-escalation rate
    budgets=[0.1, 0.2, 0.3, 0.4],  # same expected review budgets as earlier notebooks
    primary_budget=0.3,
    boot=5000,
    seeds=[0, 1, 2],                 # independent patient-fold assignments (stability of every headline number)
    primary_endpoint=dict(harm='fn', alpha=0.03, contrast='Full gate vs binary baseline'),
    ask_grid_a=[0.05, 0.10, 0.20, 0.30, 0.50],
    ask_grid_f=[0.0, 0.10, 0.25],
    lam_grid=np.geomspace(1e-5, 1.0, 240).tolist(),
    stress_seed=7,
    # regularisation of the cheap linear heads (explored on tuning patients only; see notebook text)
    crit_labels=['Edema', 'Consolidation', 'Pneumonia', 'Pneumothorax'],   # critical findings (severity >= 0.7) with their own risk guarantee
    crit_fracs=[0.75, 0.5],          # per-label target = fraction x that label's answer-everything miss rate
    ensemble_signal=True, ensemble_seeds=[0], ens_B=3,        # bootstrap-ensemble disagreement signal E (costly: run on seed 0 only)
    gate_train='nested',                 # 'dev' = gate models trained on tuning patients only; 'nested' = + out-of-fold predictions of fit patients, bagged
    head_C=dict(img=.01, ic=.001, il=.002, icl=.001, txt=.03),
    # deployment scenarios for the *clinical-context* information request (ASK-CONTEXT)
    schemes=['natural', 'mcar30', 'mcar60', 'struct40'],
    train_scheme='mcar30', primary_scheme='mcar30',
    ctx_a=0.10, ctx_f=0.05,                 # assumed cost / failure of asking the referrer for history
    ctx_grid_a=[0.02, 0.05, 0.10, 0.20, 0.30], ctx_grid_f=[0.0, 0.05, 0.20],
    lat_grid_a=[0.10, 0.25, 0.50],
)

def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def json_write(path, obj):
    path = Path(path); temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, default=str, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def atomic_npz(path, **arrays):
    path = Path(path); temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as h:
        np.savez_compressed(h, **arrays)
    temp.replace(path)

def read_npz(path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def safe_prob(p):
    return np.clip(np.asarray(p, float), EPS, 1-EPS)

def check_disjoint(data):
    for i, a in enumerate(PARTS):
        assert len(data[a]['patient']) > 0, f'Empty partition: {a}'
        for b in PARTS[i+1:]:
            assert not set(data[a]['patient']) & set(data[b]['patient']), f'Patient overlap: {a}/{b}'
    for d in data.values():
        n = len(d['patient'])
        assert d['y'].shape == d['mask'].shape == (n, len(LABELS))
        assert np.isfinite(d['image']).all() and np.isfinite(d['text']).all() and np.isfinite(d['lateral']).all()
        assert np.isin(d['y'], [0, 1]).all() and np.isin(d['mask'], [0, 1]).all()

# ------------------------------------------------------------------ outcomes
def outcome(p, y, mask):
    """Per-study outcome of answering autonomously at the fixed 0.5 cutoff (identical to V4-V13)."""
    fn = (y > .5) & (mask > .5) & (p < CUTOFF)
    fp = (y < .5) & (mask > .5) & (p >= CUTOFF)
    return dict(fn=(fn*SEVERITY).sum(1)/SEVERITY.sum(),
                severe=(fn[:, SEVERE].any(1)).astype(float),
                fp=fp.sum(1)/np.maximum((mask > .5).sum(1), 1))

# ------------------------------------------------------------------ classifier heads (numeric, cheap)
class Heads:
    """One standardiser + one regularised logistic regression per label."""
    def __init__(self, C=.1):
        self.C = C
    def fit(self, x, y, mask):
        self.scaler = StandardScaler().fit(x)
        z = self.scaler.transform(x)
        self.models = []
        for j in range(y.shape[1]):
            keep = mask[:, j] > .5
            yy = y[keep, j]
            if len(yy) < 20 or min(yy.sum(), len(yy)-yy.sum()) < 3:
                self.models.append(float((yy.sum()+1)/(len(yy)+2)))
            else:
                m = LogisticRegression(C=self.C, max_iter=300)
                m.fit(z[keep], yy)
                self.models.append(m)
        return self
    def predict(self, x):
        z = self.scaler.transform(x)
        return safe_prob(np.column_stack([np.full(len(z), m) if isinstance(m, float)
                                          else m.predict_proba(z)[:, 1] for m in self.models]))

def fit_temperature(p, y, mask, min_each=30):
    grid = np.geomspace(.25, 8, 61)
    def nll(pp, yy):
        q = safe_prob(expit(logit(safe_prob(pp))[:, None]/grid))
        return -(yy[:, None]*np.log(q)+(1-yy[:, None])*np.log1p(-q)).mean(0)
    keep = mask > .5
    tg = float(grid[np.argmin(nll(p[keep], y[keep]))]) if keep.sum() > 50 else 1.
    out = []
    for j in range(p.shape[1]):
        k = mask[:, j] > .5; yy = y[k, j]
        if min(yy.sum(), len(yy)-yy.sum()) < min_each:
            out.append(tg)
        else:
            out.append(float(grid[np.argmin(nll(p[k, j], yy))]))
    return np.array(out)

def apply_temperature(p, t):
    return safe_prob(expit(logit(safe_prob(p))/t))

def mean_brier(p, y, mask):
    return float((((p-y)**2)*mask).sum()/max(mask.sum(), 1))

def blend(pi, pt, present, w):
    return np.where(present[:, None], (1-w)*pi+w*pt, pi)

# ------------------------------------------------------------------ conformal (Mondrian / class-conditional)
def conformal_q(p, y, mask, alpha):
    q = np.ones((p.shape[1], 2))
    for j in range(p.shape[1]):
        for cls in [0, 1]:
            k = (mask[:, j] > .5) & (y[:, j] == cls)
            score = p[k, j] if cls == 0 else 1-p[k, j]
            n = int(k.sum()); rank = int(np.ceil((n+1)*(1-alpha)))
            if n and rank <= n:
                q[j, cls] = np.sort(score)[rank-1]
    return q

def pred_sets(p, q):
    return np.stack([p <= q[:, 0], (1-p) <= q[:, 1]], axis=-1)

def wilson(successes, n, z=1.959963984540054):
    if n == 0:
        return np.nan, np.nan
    p = successes/n; den = 1+z*z/n
    mid = (p+z*z/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return float(mid-half), float(mid+half)

# ------------------------------------------------------------------ the five signals
def kl(a, b):
    return a*np.log(a/b)+(1-a)*np.log((1-a)/(1-b))

def signal_blocks(p0, pi, pt, present, img, txt, aux, q):
    """U uncertainty | D cross-modal disagreement | M missing context | S clinical severity.
    Only information available BEFORE any extra data is requested. Workload (W) acts on the threshold, not here."""
    s = pred_sets(p0, q); amb = s.sum(2) == 2
    ent = -(p0*np.log(p0)+(1-p0)*np.log1p(-p0))
    U = np.column_stack([p0, ent.mean(1), amb.mean(1), (s.sum(2) == 0).mean(1), np.abs(p0-.5).min(1)])
    mid = (pi+pt)/2
    js = .5*(kl(pi, mid)+kl(pt, mid))
    pr = present.astype(float)
    D = np.column_stack([js.mean(1)*pr, js.max(1)*pr, ((pi >= .5) != (pt >= .5)).mean(1)*pr, pr])
    zi = aux['img_scaler'].transform(img)
    atyp_i = ((zi-aux['img_pca'].inverse_transform(aux['img_pca'].transform(zi)))**2).mean(1)
    zt = aux['txt_scaler'].transform(txt)
    atyp_t = ((zt-aux['txt_pca'].inverse_transform(aux['txt_pca'].transform(zt)))**2).mean(1)*pr
    M = np.column_stack([1-pr, atyp_i, atyp_t])
    S = np.column_stack([(p0*SEVERITY).sum(1)/SEVERITY.sum(), (amb*SEVERITY).max(1),
                         (p0*SEVERITY*(p0 < .5)).sum(1)/SEVERITY.sum(), (p0*SEVERITY).max(1)])
    return dict(U=U, D=D, M=M, S=S)

GATE_SETS = {'U': ['U'], 'UD': ['U', 'D'], 'UDM': ['U', 'D', 'M'], 'UDMS': ['U', 'D', 'M', 'S']}
def design(blocks, groups):
    return np.concatenate([blocks[g] for g in groups], axis=1)

def hgb(seed, n_iter=80):
    return HistGradientBoostingRegressor(max_iter=n_iter, max_leaf_nodes=7, min_samples_leaf=20,
                                         l2_regularization=2., learning_rate=.08, random_state=seed)

def patient_weights(patients):
    _, inv, counts = np.unique(patients, return_inverse=True, return_counts=True)
    w = 1/counts[inv]
    return w/w.mean()

# ------------------------------------------------------------------ guarantees: CRC and Learn-then-Test (Hoeffding-Bentkus)
def hb_pvalue(risk_hat, n, alpha):
    """Valid p-value for H0: risk > alpha, for losses bounded in [0,1] (Angelopoulos et al., Learn-then-Test)."""
    r = np.minimum(risk_hat, alpha)
    h1 = xlogy(r, r/alpha)+xlogy(1-r, (1-r)/(1-alpha))
    hoeffding = np.exp(-n*h1)
    bentkus = np.e*binom.cdf(np.ceil(n*risk_hat-1e-12), n, alpha)
    return np.minimum(1., np.minimum(hoeffding, bentkus))

def ltt_index(L, alpha, delta):
    """L: n x G calibration losses, columns ordered conservative -> aggressive. Fixed-sequence testing.
    Returns the last column of the unbroken run of rejections, or -1 if even the first fails."""
    n = L.shape[0]
    ok = hb_pvalue(L.mean(0), n, alpha) <= delta
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else L.shape[1]-1

def crc_index(L, alpha, B=1.0):
    """Conformal risk control for monotone families: largest aggressive column with (n*R+B)/(n+1) <= alpha."""
    n = L.shape[0]
    ok = (L.sum(0)+B)/(n+1) <= alpha
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else L.shape[1]-1

ESC_RATES = np.linspace(1, 0, 201)          # score gates: escalation fraction grid, conservative -> aggressive

def gate_thresholds(score_cal):
    t = np.quantile(score_cal, 1-ESC_RATES)
    t[0] = -np.inf
    return t

def gate_loss_matrix(score, harm, thresholds):
    return harm[:, None]*(score[:, None] < thresholds[None, :])

def calibrate_gate(score_cal, harm_cal, alpha, delta, method='ltt'):
    t = gate_thresholds(score_cal)
    L = gate_loss_matrix(score_cal, harm_cal, t)
    idx = ltt_index(L, alpha, delta) if method == 'ltt' else crc_index(L, alpha)
    return (-np.inf if idx < 0 else float(t[idx])), idx

# ------------------------------------------------------------------ 3-action routing (answer / escalate / ask)
def route_lagrange(c0hat, gainhat, eligible, lam, e, f, a, allow_ask):
    c0 = np.maximum(c0hat, 0.)
    c1 = np.maximum(c0hat-gainhat, 0.)
    obj = np.stack([c0, e*c0+lam,
                    np.where(allow_ask & eligible, (1-f)*c1+f*e*c0+lam*(a+f), np.inf)], axis=1)
    return obj.argmin(1)                    # 0 answer, 1 escalate, 2 ask

def realised(act, c0, c1, e, f):
    return np.where(act == 0, c0, np.where(act == 1, e*c0, (1-f)*c1+f*e*c0))

def workload_of(act, a, f):
    return np.where(act == 1, 1., np.where(act == 2, a+f, 0.))

def pick_lambda(c0hat, gainhat, eligible, e, f, a, allow_ask, budget, lam_grid):
    """Smallest price of workload whose label-free workload fits the budget."""
    for lam in lam_grid:
        act = route_lagrange(c0hat, gainhat, eligible, lam, e, f, a, allow_ask)
        if workload_of(act, a, f).mean() <= budget+1e-9:
            return lam, act
    return lam_grid[-1], act

def topk_escalate(score, budget):
    n = len(score); k = int(np.floor(budget*n+1e-9))
    act = np.zeros(n, int)
    if k > 0:
        act[np.argsort(-score, kind='stable')[:k]] = 1
    return act

# ------------------------------------------------------------------ statistics
def boot_ci(x, reps=2000, seed=0):
    """Percentile 95% interval of the mean, a two-sided bootstrap p-value and the per-unit sd (for power calculations)."""
    x = np.asarray(x, float); n = len(x); rng = np.random.default_rng(seed)
    vals = np.empty(reps)
    for s in range(0, reps, 200):
        m = min(200, reps-s)
        vals[s:s+m] = x[rng.integers(0, n, (m, n))].mean(1)
    lo, hi = np.quantile(vals, [.025, .975])
    p = min(1., 2*(min((vals <= 0).sum(), (vals >= 0).sum())+1)/(reps+1))
    sd = float(x.std(ddof=1)) if n > 1 else float('nan')
    return dict(mean=float(x.mean()), lo=float(lo), hi=float(hi), n=n, p=float(p), sd=sd, mde80=float(2.8*sd/np.sqrt(n)) if n > 1 else float('nan'))

def holm(pvals):
    """Holm-Bonferroni adjusted p-values (family-wise error control)."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.
    for rank, i in enumerate(order):
        run = max(run, min(1., (m-rank)*p[i])); adj[i] = run
    return adj

def verdict(ci, lower_is_better=True):
    if ci['hi'] < 0:
        return 'better' if lower_is_better else 'worse'
    if ci['lo'] > 0:
        return 'worse' if lower_is_better else 'better'
    return 'inconclusive'

def auc_delta_ci(y, a, b, reps=1000, seed=0):
    y = np.asarray(y); n = len(y); rng = np.random.default_rng(seed); vals = []
    for _ in range(reps):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2:
            continue
        vals.append([roc_auc_score(y[i], b[i])-roc_auc_score(y[i], a[i]),
                     average_precision_score(y[i], b[i])-average_precision_score(y[i], a[i])])
    vals = np.asarray(vals)
    out = {}
    for j, (name, fn) in enumerate([('auroc', roc_auc_score), ('auprc', average_precision_score)]):
        lo, hi = np.quantile(vals[:, j], [.025, .975])
        out[name] = dict(a=float(fn(y, a)), b=float(fn(y, b)), delta=float(fn(y, b)-fn(y, a)), lo=float(lo), hi=float(hi))
    return out


# ------------------------------------------------------------------ four-action routing: answer / escalate / ask-context / ask-lateral
def route4(c0hat, gc, gl, ctx_ok, lat_ok, lam, e, p):
    c0 = np.maximum(c0hat, 0.)
    cc = np.maximum(c0-gc, 0.); cl = np.maximum(c0-gl, 0.)
    obj = np.stack([c0, e*c0+lam,
                    np.where(p['allow_ctx'] & ctx_ok, (1-p['f_c'])*cc+p['f_c']*e*c0+lam*(p['a_c']+p['f_c']), np.inf),
                    np.where(p['allow_lat'] & lat_ok, (1-p['f_l'])*cl+p['f_l']*e*c0+lam*(p['a_l']+p['f_l']), np.inf)], axis=1)
    return obj.argmin(1)                    # 0 answer, 1 escalate, 2 ask context, 3 ask lateral

def realised4(act, c0, cc, cl, e, p):
    return np.select([act == 0, act == 1, act == 2, act == 3],
                     [c0, e*c0, (1-p['f_c'])*cc+p['f_c']*e*c0, (1-p['f_l'])*cl+p['f_l']*e*c0])

def workload4(act, p):
    return np.select([act == 1, act == 2, act == 3], [1., p['a_c']+p['f_c'], p['a_l']+p['f_l']], 0.)

def pick_lambda4(c0hat, gc, gl, ctx_ok, lat_ok, e, p, budget, lam_grid):
    for lam in lam_grid:
        act = route4(c0hat, gc, gl, ctx_ok, lat_ok, lam, e, p)
        if workload4(act, p).mean() <= budget+1e-9:
            return lam, act
    return lam_grid[-1], act

def hash_uniform(strings, salt):
    return np.array([int(hashlib.sha256(f'{salt}:{s}'.encode()).hexdigest()[:12], 16)/16**12 for s in strings])


# ------------------------------------------------------------------ learning-to-defer baseline (surrogate-loss gate)
from scipy.optimize import minimize
class L2DGate:
    """Learning-to-defer baseline in the spirit of Madras et al. / Mozannar & Sontag: a gate g(x) trained to minimise the expected team cost
    E[(1-g)*c + g*(e*c + lam)] for a fixed classifier and an expert that leaves a residual e of the loss. It is trained directly on the
    deferral objective (no intermediate loss regression). To give it a fair function class it uses all degree-2 feature interactions."""
    def fit(self, X, c, e, lam, w=None, reg=3e-3):
        from sklearn.preprocessing import PolynomialFeatures
        self.poly = PolynomialFeatures(2, include_bias=False)
        Z0 = self.poly.fit_transform((X-X.mean(0))/(X.std(0)+1e-8)); self.mu0, self.sd0 = X.mean(0), X.std(0)+1e-8
        self.mu = Z0.mean(0); self.sd = Z0.std(0)+1e-8; Z = (Z0-self.mu)/self.sd
        w = np.ones(len(X)) if w is None else w/np.mean(w)
        delta = (e*c+lam)-c
        def f(theta):
            g = expit(Z@theta[:-1]+theta[-1])
            loss = np.mean(w*(c+g*delta))+reg*np.sum(theta[:-1]**2)
            dz = w*g*(1-g)*delta/len(Z)
            return loss, np.concatenate([Z.T@dz+2*reg*theta[:-1], [dz.sum()]])
        self.theta = minimize(f, np.zeros(Z.shape[1]+1), jac=True, method='L-BFGS-B', options=dict(maxiter=400)).x
        return self
    def score(self, X):
        Z = (self.poly.transform((X-self.mu0)/self.sd0)-self.mu)/self.sd
        return Z@self.theta[:-1]+self.theta[-1]


def multi_ltt_index(Ls, alphas, delta):
    """Fixed-sequence Learn-then-Test for SEVERAL risks at once: a column is accepted only if every risk's Hoeffding-Bentkus p-value is <= delta
    (delta already includes the Bonferroni split across risks). Returns the last column of the unbroken accepted run, or -1."""
    n = Ls[0].shape[0]
    ok = np.ones(Ls[0].shape[1], bool)
    for L, a in zip(Ls, alphas):
        ok &= hb_pvalue(L.mean(0), n, a) <= delta
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else Ls[0].shape[1]-1


## 1. Load and verify the cached snapshot
Checks manifest hashes, patient disjointness, array shapes and cohort counts against the completed run, and stops if any check fails.

In [ ]:
frozen = json.loads((RUN_DIR/'frozen_manifest.json').read_text())
original_cfg = json.loads((RUN_DIR/'config.json').read_text())
encoder = json.loads((RUN_DIR/'encoder_manifest.json').read_text())
CFG = dict(frozen['config'])
print('Encoder:', encoder.get('image_encoder'), '| dim', encoder.get('image_dim'))
if encoder.get('image_encoder') != 'biomedclip':
    raise ValueError('This notebook expects the completed BiomedCLIP snapshot.')

snapshot = RUN_DIR/'persistent_encoded_data_PRIVATE.pkl'
with snapshot.open('rb') as handle:
    data = pickle.load(handle)
check_disjoint(data)
digest = hashlib.sha256()
rows = []
for part in PARTS:
    d = data[part]
    if stable_hash(sorted(d['case'].tolist())) != frozen['partitions'][part]:
        raise ValueError(f'{part}: snapshot differs from the completed manifest.')
    if len(set(d['case'])) != len(d['case']):
        raise ValueError('Duplicate study identifiers in '+part)
    digest.update(part.encode())
    digest.update(json.dumps(sorted(d['case'].tolist())).encode())
    digest.update(str((d['image'].shape, d['text'].shape, float(np.asarray(d['image'], float).sum()))).encode())
    rows.append(dict(partition=part, studies=len(d['case']), patients=len(set(d['patient'])),
                     lateral_eligible=int(d['eligible'].sum()), context_present=int(d['present'].sum())))
INPUT_FINGERPRINT = digest.hexdigest()
saved = pd.read_csv(RUN_DIR/'cohort_summary.csv')
pd.testing.assert_frame_equal(pd.DataFrame(rows).reset_index(drop=True), saved.reset_index(drop=True), check_dtype=False)
print('Snapshot verified against the completed run (hashes, disjoint patients, shapes, counts).')
display(pd.DataFrame(rows))


## 2. Patient groups, evaluation units, cross-fitting folds and missing-history scenarios
Six patient groups rotate through roles: three fit the classifier heads, one tunes (temperature, conformal quantiles and referral models), one calibrates the referral threshold and one evaluates. The scenarios determine which patients lack clinical history at triage; the assignment is hash-based and does not use labels.

In [ ]:
POOL_KEYS = ['case', 'patient', 'y', 'mask', 'image', 'text', 'lateral', 'eligible', 'present']
P = {k: np.concatenate([np.asarray(data[p][k]) for p in PARTS]) for k in POOL_KEYS}
P['case'] = P['case'].astype(str); P['patient'] = P['patient'].astype(str)
P['eligible'] = P['eligible'].astype(bool); P['present'] = P['present'].astype(bool)
P['y'] = P['y'].astype(float); P['mask'] = P['mask'].astype(float)
P['image'] = P['image'].astype(np.float32); P['text'] = P['text'].astype(np.float32)
P['lateral'] = P['lateral'].astype(np.float32)
N = len(P['case'])
assert len(set(P['case'])) == N, 'case identifiers must be unique across partitions'
del data; gc.collect()

# One label-blind "unit" study per patient: the study with the smallest hash of its identifier.
case_key = np.array([hashlib.sha256(('unit:'+c).encode()).hexdigest() for c in P['case']])
order = np.lexsort((case_key, P['patient']))
first = np.r_[True, P['patient'][order][1:] != P['patient'][order][:-1]]
unit = np.zeros(N, bool); unit[order[first]] = True
P['unit'] = unit

# Label-blind variables used to SIMULATE that clinical context (indication/history) is not available at triage.
P['u_ctx'] = hash_uniform(P['case'], 'ctx')
_sc = StandardScaler().fit(P['image'][::3])
_pc = PCA(n_components=1, random_state=0).fit(_sc.transform(P['image'][::3]))
_z = _pc.transform(_sc.transform(P['image']))[:, 0]
_s = 0.8*(np.argsort(np.argsort(_z))/len(_z)) + 0.2*hash_uniform(P['case'], 'struct')
P['struct'] = np.argsort(np.argsort(_s))/len(_s)          # exactly uniform, but driven by the image-embedding direction (structured, non-random)
def ctx_available(rows, scheme):
    pres = P['present'][rows]
    if scheme == 'natural':
        return pres.copy()
    m = int(re.sub(r'\D', '', scheme))/100
    if scheme.startswith('mcar'):
        return pres & (P['u_ctx'][rows] >= m)
    if scheme.startswith('struct'):
        return pres & (P['struct'][rows] < 1-m)
    raise KeyError(scheme)

# Patient-grouped fold assignment: hash order, then balanced round robin. Different `seed_index` values give independent assignments.
patients = np.unique(P['patient'])
def assign_folds(seed_index):
    u = np.array([int(hashlib.sha256(f"{PLAN['seed']+1000*seed_index}:{p}".encode()).hexdigest()[:12], 16) for p in patients])
    fop = dict(zip(patients[np.argsort(u)], np.arange(len(patients)) % PLAN['folds']))
    P['fold'] = np.array([fop[p] for p in P['patient']])
assign_folds(0)
K = PLAN['folds']
def roles_for(k):
    f = P['fold']
    return dict(ev=f == k, cal=f == (k+1) % K, dev=f == (k+2) % K,
                fit=~np.isin(f, [k, (k+1) % K, (k+2) % K]))

print(f'Pooled {N:,} studies from {len(patients):,} patients; {int(unit.sum()):,} unit studies (one per patient).')
print('Unit studies with lateral view available:', f"{P['eligible'][unit].mean():.1%}",
      '| context present:', f"{P['present'][unit].mean():.1%}")
print('Patients per fold:', np.bincount(P['fold'][unit]))
print('Why pooling: the original test partition held only 184 patients (2,000 studies, ~11 per patient).')
print('Cross-fitting gives every patient exactly one out-of-sample evaluation by models that never saw them.')

# Costs used for the Ask action; taken from the frozen configuration, never tuned here.
ASK_A = float(CFG.get('ask_workload', 0.25)); ASK_F = float(CFG.get('retrieval_failure', 0.15))
EXPERT_E = float(CFG.get('expert_residual', 0.10))
CTX_A, CTX_F = PLAN['ctx_a'], PLAN['ctx_f']
print(f'Assumed: lateral request a={ASK_A}, failure f={ASK_F} (from frozen config); context request a={CTX_A}, failure f={CTX_F}; expert residual e={EXPERT_E}')
for sch in PLAN['schemes']:
    print(f'  scheme {sch:9s}: context available for {ctx_available(np.flatnonzero(unit), sch).mean():.1%} of patients at triage')


## 3. Cross-fitted pipeline (heads, referral signals, referral models, scenarios)
Classifier heads are regularized logistic regressions on cached embeddings; regularization strengths were explored on tuning patients from the pooled development data. Referral models are gradient-boosted regressors trained on tuning patients in the scenario with 30% of histories withheld at random.

In [ ]:
identity = stable_hash({'plan': PLAN, 'input': INPUT_FINGERPRINT, 'ask': [ASK_A, ASK_F, EXPERT_E]})[:20]
OUT_ROOT = RUN_DIR/'methodology_v19'/identity
OUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT = OUT_ROOT/'seed0'; OUT.mkdir(exist_ok=True)
json_write(OUT_ROOT/'plan_LOCKED.json', {**PLAN, 'input_fingerprint': INPUT_FINGERPRINT,
                                    'ask_a': ASK_A, 'ask_f': ASK_F, 'expert_e': EXPERT_E})
print('V19 output folder (new; earlier results untouched):', OUT_ROOT)
TZ = (P['text']*P['present'][:, None]).astype(np.float32)          # context embedding, zero when absent
HC = PLAN['head_C']

def make_aux(Pimg, Ptxt, seed):
    rng = np.random.default_rng(seed)
    def pca_for(x):
        if len(x) > 8000:
            x = x[rng.choice(len(x), 8000, replace=False)]
        sc = StandardScaler().fit(x)
        comps = int(min(32, max(1, x.shape[1]//2), len(x)-1))
        return sc, PCA(n_components=comps, random_state=seed).fit(sc.transform(x))
    isc, ipca = pca_for(Pimg); tsc, tpca = pca_for(Ptxt)
    return dict(img_scaler=isc, img_pca=ipca, txt_scaler=tsc, txt_pca=tpca)

def score_dict(reg, blocks):
    out = {}
    for F, groups in GATE_SETS.items():
        X = design(blocks, groups)
        out['sc_fn_'+F] = reg['fn_'+F].predict(X)
        out['sc_sv_'+F] = reg['sv_'+F].predict(X)
        out['c0hat_'+F] = np.maximum(reg['c0_'+F].predict(X), 0.)
    Xf = design(blocks, GATE_SETS['UDMS'])
    out['gc'] = reg['gc'].predict(Xf); out['gl'] = reg['gl'].predict(Xf)
    for F in ['U', 'UDMS']:
        out['sc_l2d_'+F] = reg['l2d_'+F].score(design(blocks, GATE_SETS[F]))
    out['sc_entropy'] = blocks['U'][:, 14]
    out['sc_ambiguity'] = blocks['U'][:, 15]
    out['sc_sevharm'] = blocks['S'][:, 2]
    return out

GATE_SETS_E = {'UE': ['U', 'E'], 'UDMSE': ['U', 'D', 'M', 'S', 'E']}

class Bag:
    def __init__(self, members):
        self.members = members
    def predict(self, X):
        return np.mean([m.predict(X) for m in self.members], axis=0)

class _Zero:
    def predict(self, X):
        return np.zeros(len(X))

def fit_fold(k, verbose=True):
    path, bpath = OUT/f'fold{k}.npz', OUT/f'fold{k}_bundle_PRIVATE.pkl'
    if path.is_file() and bpath.is_file():
        if verbose: print(f'fold {k}: reusing checkpoint')
        return
    t_start = time.time()
    R = roles_for(k); ix = {r: np.flatnonzero(m) for r, m in R.items()}
    fit, dev = ix['fit'], ix['dev']
    cal = ix['cal'][unit[ix['cal']]]; ev = ix['ev'][unit[ix['ev']]]
    seed = PLAN['seed']+k+1000*CUR_SEED
    y_all, m_all = P['y'], P['mask']
    # ---- 1) classifier heads on the fit patients only (strongly regularised linear heads on cached embeddings)
    def fit_heads(rows, light=False):
        Hh = {}
        Hh['img'] = Heads(HC['img']).fit(P['image'][rows], y_all[rows], m_all[rows])
        Hh['ic'] = Heads(HC['ic']).fit(np.hstack([P['image'][rows], TZ[rows]]), y_all[rows], m_all[rows])
        if light:
            return Hh
        rp = rows[P['present'][rows]]
        Hh['txt'] = Heads(HC['txt']).fit(TZ[rp], y_all[rp], m_all[rp])
        rl = rows[P['eligible'][rows]]
        Hh['il'] = Heads(HC['il']).fit(np.hstack([P['image'][rl], P['lateral'][rl]]), y_all[rl], m_all[rl])
        Hh['icl'] = Heads(HC['icl']).fit(np.hstack([P['image'][rl], TZ[rl], P['lateral'][rl]]), y_all[rl], m_all[rl])
        return Hh
    H = fit_heads(fit)
    fp_ = fit[P['present'][fit]]
    def raw_preds(rows, Hh=None):
        Hh = H if Hh is None else Hh
        img, tz, lat, el = P['image'][rows], TZ[rows], P['lateral'][rows], P['eligible'][rows]
        out = dict(img=Hh['img'].predict(img), ic=Hh['ic'].predict(np.hstack([img, tz])), txt=Hh['txt'].predict(tz))
        out['il'] = out['img'].copy(); out['icl'] = out['ic'].copy()
        if el.any():
            out['il'][el] = Hh['il'].predict(np.hstack([img[el], lat[el]]))
            out['icl'][el] = Hh['icl'].predict(np.hstack([img[el], tz[el], lat[el]]))
        return out
    # ---- 2) temperatures (tuning patients only)
    raw_d = raw_preds(dev); yd, md = y_all[dev], m_all[dev]
    pres_d, el_d = P['present'][dev], P['eligible'][dev]
    sel = dict(img=np.ones(len(dev), bool), ic=pres_d, txt=pres_d, il=el_d, icl=el_d & pres_d)
    TEMP = {}
    for key, s in sel.items():
        TEMP[key] = fit_temperature(raw_d[key][s], yd[s], md[s]) if s.sum() > 60 else np.ones(len(LABELS))
    def cal_preds(rows):
        raw = raw_preds(rows)
        return {key: apply_temperature(raw[key], TEMP[key]) for key in raw}
    def states(rows, pr, scheme):
        avail = ctx_available(rows, scheme); pres = P['present'][rows]; el = P['eligible'][rows]
        p0 = np.where(avail[:, None], pr['ic'], pr['img'])
        pc = np.where(pres[:, None], pr['ic'], pr['img'])                  # after the context request succeeds
        pl = np.where(avail[:, None], pr['icl'], pr['il'])                 # after the lateral request succeeds
        return avail, pres & ~avail, el, p0, pc, pl
    # ---- 2b) bootstrap-ensemble disagreement signal E (epistemic uncertainty): spread of calibrated predictions across heads trained on
    #          patient-level bootstrap resamples. Only computed for the seeds listed in PLAN['ensemble_seeds'].
    use_E = bool(PLAN['ensemble_signal']) and CUR_SEED in PLAN['ensemble_seeds']
    def boot_rows(rows, b):
        pr_ = P['patient'][rows]; order = np.argsort(pr_, kind='stable'); rs, ps = rows[order], pr_[order]
        uniq, start, cnt = np.unique(ps, return_index=True, return_counts=True)
        rng_ = np.random.default_rng(seed+500+b); pick = rng_.integers(0, len(uniq), len(uniq))
        return np.concatenate([rs[start[i]:start[i]+cnt[i]] for i in pick])
    def make_members(rows):
        return [fit_heads(boot_rows(rows, b), light=True) for b in range(PLAN['ens_B'])]
    def ens_E(rows, members, avail):
        img, tz = P['image'][rows], TZ[rows]; ps = []
        for Hb in members:
            pi_ = apply_temperature(Hb['img'].predict(img), TEMP['img']); pc_ = apply_temperature(Hb['ic'].predict(np.hstack([img, tz])), TEMP['ic'])
            ps.append(np.where(avail[:, None], pc_, pi_))
        sd = np.std(np.stack(ps), axis=0)
        return np.column_stack([sd.mean(1), sd.max(1), (sd*SEVERITY).sum(1)/SEVERITY.sum()])
    members_full = make_members(fit) if use_E else None
    # ---- 3) conformal quantiles, PCA objects, gate regressors (tuning patients, training scheme only)
    pr_d = cal_preds(dev)
    avail_d, ctxok_d, el_d, p0_d, pc_d, pl_d = states(dev, pr_d, PLAN['train_scheme'])
    dev_u = unit[dev]
    q = conformal_q(p0_d[dev_u], yd[dev_u], md[dev_u], PLAN['conformal_alpha'])
    if PLAN['gate_train'] == 'nested':
        # Gate models are trained on the tuning patients PLUS out-of-fold predictions for half of the fit patients (heads re-fitted without
        # each inner third), i.e. about 2.5x more patients. The other half of the fit patients fits the PCA objects, so atypicality
        # features of every gate-training row are out-of-sample.
        pat_fit = P['patient'][fit]
        grp = np.minimum((hash_uniform(pat_fit, f'inner{CUR_SEED}')*3).astype(int), 2)
        halfA = hash_uniform(pat_fit, f'half{CUR_SEED}') < 0.5
        oof = {key: np.zeros((len(fit), len(LABELS))) for key in ['img', 'ic', 'txt', 'il', 'icl']}
        E_oof = np.zeros((len(fit), 3)); avail_fit = ctx_available(fit, PLAN['train_scheme'])
        for g in range(3):
            te = np.flatnonzero(grp == g); rg = raw_preds(fit[te], fit_heads(fit[grp != g]))
            for key in oof:
                oof[key][te] = rg[key]
            if use_E:
                teB = te[~halfA[te]]
                if len(teB):
                    E_oof[teB] = ens_E(fit[teB], make_members(fit[grp != g]), avail_fit[teB])
        pr_fit = {key: apply_temperature(oof[key], TEMP[key]) for key in oof}
        fA = fit[halfA]; sel_B = ~halfA
        aux = make_aux(P['image'][fA], TZ[fA[P['present'][fA]]], seed)
        rows_tr = np.concatenate([dev, fit[sel_B]])
        pr_tr = {key: np.concatenate([pr_d[key], pr_fit[key][sel_B]]) for key in pr_d}
    else:
        aux = make_aux(P['image'][fit], TZ[fp_], seed)
        rows_tr, pr_tr = dev, pr_d
    avail_t, ctxok_t, el_t, p0_t, pc_t, pl_t = states(rows_tr, pr_tr, PLAN['train_scheme'])
    bd = signal_blocks(p0_t, pr_tr['img'], pr_tr['txt'], avail_t, P['image'][rows_tr], TZ[rows_tr], aux, q)
    if use_E:
        E_dev = ens_E(dev, members_full, avail_d)
        bd['E'] = np.vstack([E_dev, E_oof[~halfA]]) if PLAN['gate_train'] == 'nested' else E_dev
    yt, mt = y_all[rows_tr], m_all[rows_tr]
    o0, oc, ol = outcome(p0_t, yt, mt), outcome(pc_t, yt, mt), outcome(pl_t, yt, mt)
    cost = lambda o: o['fn']+PLAN['fp_weight']*o['fp']
    c0, cc, cl = cost(o0), cost(oc), cost(ol)
    pat_t = P['patient'][rows_tr]
    w = patient_weights(pat_t); reg = {}
    def fit_reg(sd, X, y, wt):
        if PLAN['gate_train'] != 'nested':
            return hgb(sd).fit(X, y, sample_weight=wt)
        rng_ = np.random.default_rng(sd); members = []
        for i in range(3):                              # bagging: three fits on random 80% subsamples of rows
            keep = rng_.random(len(X)) < 0.8
            members.append(hgb(sd+17*i).fit(X[keep], y[keep], sample_weight=wt[keep]))
        return Bag(members)
    for F, groups in GATE_SETS.items():
        X = design(bd, groups)
        reg['fn_'+F] = fit_reg(seed, X, o0['fn'], w)
        reg['sv_'+F] = fit_reg(seed+1, X, o0['severe'], w)
        reg['c0_'+F] = fit_reg(seed+2, X, c0, w)
    if use_E:
        for F, groups in GATE_SETS_E.items():
            X = design(bd, groups)
            reg['fn_'+F] = fit_reg(seed+20, X, o0['fn'], w); reg['sv_'+F] = fit_reg(seed+21, X, o0['severe'], w)
    CRIT = PLAN['crit_labels']; crit_j = [LABELS.index(l) for l in CRIT]
    for lab, j in zip(CRIT, crit_j):
        miss_t = ((yt[:, j] > .5) & (mt[:, j] > .5) & (p0_t[:, j] < CUTOFF)).astype(float)
        for F in ['U', 'UDMS']:
            reg[f'mj_{lab}_{F}'] = fit_reg(seed+30+j, design(bd, GATE_SETS[F]), miss_t, w)
    Xf = design(bd, GATE_SETS['UDMS'])
    lam_l2d = float(np.quantile((1-EXPERT_E)*c0, .70))
    for F in ['U', 'UDMS']:
        reg['l2d_'+F] = L2DGate().fit(design(bd, GATE_SETS[F]), c0, EXPERT_E, lam_l2d, w)
    reg['gc'] = fit_reg(seed+3, Xf[ctxok_t], (c0-cc)[ctxok_t], patient_weights(pat_t[ctxok_t])) if ctxok_t.sum() > 60 else _Zero()
    reg['gl'] = fit_reg(seed+4, Xf[el_t], (c0-cl)[el_t], patient_weights(pat_t[el_t])) if el_t.sum() > 60 else _Zero()
    # ---- 4) calibration + evaluation patients, every deployment scheme
    arrays = dict(q=q)
    for role, rows in [('cal', cal), ('ev', ev)]:
        pr = cal_preds(rows)
        arrays.update({f'{role}_case': P['case'][rows], f'{role}_patient': P['patient'][rows],
                       f'{role}_eligible': P['eligible'][rows], f'{role}_present': P['present'][rows],
                       f'{role}_y': y_all[rows], f'{role}_mask': m_all[rows],
                       f'{role}_pr_img': pr['img'], f'{role}_pr_ic': pr['ic'], f'{role}_pr_il': pr['il'], f'{role}_pr_icl': pr['icl']})
        for scheme in PLAN['schemes']:
            avail, ctxok, el, p0, pc, pl = states(rows, pr, scheme)
            b = signal_blocks(p0, pr['img'], pr['txt'], avail, P['image'][rows], TZ[rows], aux, q)
            a0, ac, al = outcome(p0, y_all[rows], m_all[rows]), outcome(pc, y_all[rows], m_all[rows]), outcome(pl, y_all[rows], m_all[rows])
            if use_E:
                b['E'] = ens_E(rows, members_full, avail)
            extra = {}
            for lab, j in zip(CRIT, crit_j):
                extra[f'miss_{lab}'] = ((y_all[rows][:, j] > .5) & (m_all[rows][:, j] > .5) & (p0[:, j] < CUTOFF)).astype(float)
                for F in ['U', 'UDMS']:
                    extra[f'scj_{lab}_{F}'] = reg[f'mj_{lab}_{F}'].predict(design(b, GATE_SETS[F]))
            if use_E:
                for F, groups in GATE_SETS_E.items():
                    extra['sc_fn_'+F] = reg['fn_'+F].predict(design(b, groups)); extra['sc_sv_'+F] = reg['sv_'+F].predict(design(b, groups))
            rec = dict(avail=avail, ctx_ok=ctxok, lat_ok=el, p0=p0, dmean=b['D'][:, 0], **extra,
                       fn0=a0['fn'], fp0=a0['fp'], sev0=a0['severe'], fnc=ac['fn'], fpc=ac['fp'], sevc=ac['severe'],
                       fnl=al['fn'], fpl=al['fp'], sevl=al['severe'], **score_dict(reg, b))
            arrays.update({f'{role}_{scheme}_{key}': val for key, val in rec.items()})
    # ---- 5) stress test for Signal 2: pair every X-ray with ANOTHER patient's clinical context (primary scheme)
    rng = np.random.default_rng(PLAN['stress_seed']+k)
    order_ = rng.permutation(len(ev)); partner = np.empty(len(ev), int); partner[order_] = np.roll(order_, -1)
    avail = ctx_available(ev, PLAN['primary_scheme'])
    tz_m = TZ[ev][partner]; pres_m = P['present'][ev][partner]
    img_ev = P['image'][ev]
    pic_m = apply_temperature(H['ic'].predict(np.hstack([img_ev, tz_m])), TEMP['ic'])
    ptx_m = apply_temperature(H['txt'].predict(tz_m), TEMP['txt'])
    pr_ev = cal_preds(ev)
    avail_m = avail & pres_m
    p0_m = np.where(avail_m[:, None], pic_m, pr_ev['img'])
    b_m = signal_blocks(p0_m, pr_ev['img'], ptx_m, avail_m, img_ev, tz_m, aux, q)
    sm = score_dict(reg, b_m)
    arrays['mis_dmean'] = b_m['D'][:, 0]; arrays['mis_fn0'] = outcome(p0_m, y_all[ev], m_all[ev])['fn']
    arrays['mis_avail'] = avail_m
    for F in GATE_SETS:
        arrays['mis_sc_fn_'+F] = sm['sc_fn_'+F]
    with bpath.open('wb') as h:
        pickle.dump(dict(H=H, TEMP=TEMP, q=q, aux=aux, reg=reg), h)
    atomic_npz(path, **arrays)
    if verbose:
        print(f'fold {k}: fit {len(fit):,} / dev {len(dev):,} / cal {len(cal):,} / eval {len(ev):,} studies; {time.time()-t_start:.0f}s', flush=True)

from threadpoolctl import threadpool_limits
CUR_SEED = 0
def run_seed(s):
    """Select fold assignment `s` and make sure its six folds exist (checkpointed under seed<s>/)."""
    global OUT, CUR_SEED
    CUR_SEED = s; assign_folds(s); OUT = OUT_ROOT/f'seed{s}'; OUT.mkdir(exist_ok=True)
    with threadpool_limits(limits=max(1, (os.cpu_count() or 2))):
        for k in range(K):
            fit_fold(k)
run_seed(0)
print('Seed 0 folds complete (primary analysis). Stability seeds run later; re-running only reloads checkpoints.')


## 4. Classifier discrimination, information value and conformal coverage

In [ ]:
BASE_KEYS = ['case', 'patient', 'eligible', 'present', 'y', 'mask', 'pr_img', 'pr_ic', 'pr_il', 'pr_icl']
SCHEME_KEYS = ['avail', 'ctx_ok', 'lat_ok', 'p0', 'dmean', 'fn0', 'fp0', 'sev0', 'fnc', 'fpc', 'sevc', 'fnl', 'fpl', 'sevl',
               'gc', 'gl', 'sc_entropy', 'sc_ambiguity', 'sc_sevharm', 'sc_l2d_U', 'sc_l2d_UDMS'] +               [f'{a}_{F}' for F in GATE_SETS for a in ['sc_fn', 'sc_sv', 'c0hat']] + \
              [f'miss_{l}' for l in PLAN['crit_labels']] + [f'scj_{l}_{F}' for l in PLAN['crit_labels'] for F in ['U', 'UDMS']]
def pooled(role, key):
    return np.concatenate([F[f'{role}_{key}'] for F in FOLDS])
def load_seed():
    """(Re)bind the pooled out-of-sample arrays of the CURRENT seed's folds (global OUT)."""
    global FOLDS, EVB, NE, FS
    FOLDS = [read_npz(OUT/f'fold{k}.npz') for k in range(K)]
    EVB = {key: pooled('ev', key) for key in BASE_KEYS}
    EVB['fold'] = np.concatenate([np.full(len(F['ev_case']), k) for k, F in enumerate(FOLDS)])
    NE = len(EVB['case'])
    assert len(set(EVB['patient'])) == NE, 'each evaluation patient must appear exactly once'
    FS = [np.flatnonzero(EVB['fold'] == k) for k in range(K)]
load_seed()
def build_EV(scheme):
    ev = dict(EVB)
    ev.update({key: pooled('ev', f'{scheme}_{key}') for key in SCHEME_KEYS})
    return ev
def get_cal(k, scheme):
    F = FOLDS[k]
    return {key: F[f'cal_{scheme}_{key}'] for key in SCHEME_KEYS}
PRIMARY = PLAN['primary_scheme']
EV = build_EV(PRIMARY)
RESULTS = {}      # everything that ends up in the report
TABLES = {}       # DataFrames exported to Excel/CSV
FPW = PLAN['fp_weight']
print(f'{NE:,} patients evaluated out-of-sample (the original test partition held 184). Primary deployment scenario: {PRIMARY}.')
base = dict(answer_all_fn=float(EV['fn0'].mean()), answer_all_severe=float(EV['sev0'].mean()), answer_all_fp=float(EV['fp0'].mean()))
RESULTS['answer_all_primary_scheme'] = base
print('Answer-everything risk, primary scenario (no oversight):', {k: round(v, 4) for k, v in base.items()})

# ---------------------------------------------------------------- A. classifier quality and the value of the two information requests
def ece_of(p, yy, bins=10):
    e = 0.; edges = np.linspace(0, 1, bins+1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        b = (p >= lo) & (p < hi) if hi < 1 else (p >= lo) & (p <= hi)
        if b.any():
            e += b.mean()*abs(p[b].mean()-yy[b].mean())
    return e
def auc_or_nan(yy, pp):
    return roc_auc_score(yy, pp) if min(yy.sum(), len(yy)-yy.sum()) >= 20 else np.nan
rows = []
for j, label in enumerate(LABELS):
    k = EVB['mask'][:, j] > .5; yy = EVB['y'][k, j]
    kp = k & EVB['present']; ke = k & EVB['eligible']
    rows.append(dict(label=label, n=int(k.sum()), positives=int(yy.sum()),
                     auroc_image_only=auc_or_nan(yy, EVB['pr_img'][k, j]),
                     auroc_image_plus_context=auc_or_nan(EVB['y'][kp, j], EVB['pr_ic'][kp, j]),
                     auroc_image_only_on_present=auc_or_nan(EVB['y'][kp, j], EVB['pr_img'][kp, j]),
                     auroc_image_plus_lateral=auc_or_nan(EVB['y'][ke, j], EVB['pr_il'][ke, j]),
                     auroc_image_only_on_eligible=auc_or_nan(EVB['y'][ke, j], EVB['pr_img'][ke, j]),
                     ece_image_plus_context=ece_of(EVB['pr_ic'][kp, j], EVB['y'][kp, j])))
CLS = pd.DataFrame(rows)
CLS['gain_context'] = CLS.auroc_image_plus_context-CLS.auroc_image_only_on_present
CLS['gain_lateral'] = CLS.auroc_image_plus_lateral-CLS.auroc_image_only_on_eligible
TABLES['classifier_per_label'] = CLS
RESULTS['classifier'] = dict(macro_auroc_image_only=float(CLS.auroc_image_only.mean()),
                             macro_auroc_with_context=float(CLS.auroc_image_plus_context.mean()),
                             macro_gain_context=float(CLS.gain_context.mean()), macro_gain_lateral=float(CLS.gain_lateral.mean()),
                             macro_ece_with_context=float(CLS.ece_image_plus_context.mean()))
display(CLS.round(3))
print('Information value (macro AUROC):', {k: round(v, 4) for k, v in RESULTS['classifier'].items()})

# realised cost value of each information request (all patients who could be asked), patient-bootstrap intervals
c_img = EV['fn0']; cost_of = lambda fn, fp: fn+FPW*fp
fn_i, fp_i = outcome(EVB['pr_img'], EVB['y'], EVB['mask']), None
o_img = outcome(EVB['pr_img'], EVB['y'], EVB['mask']); o_ic = outcome(EVB['pr_ic'], EVB['y'], EVB['mask'])
o_il = outcome(EVB['pr_il'], EVB['y'], EVB['mask']); o_icl = outcome(EVB['pr_icl'], EVB['y'], EVB['mask'])
pr_ = EVB['present']; el_ = EVB['eligible']
gain_ctx = (cost_of(o_img['fn'], o_img['fp'])-cost_of(o_ic['fn'], o_ic['fp']))[pr_]
gain_lat = (cost_of(o_img['fn'], o_img['fp'])-cost_of(o_il['fn'], o_il['fp']))[el_]
VOI = dict(context=boot_ci(gain_ctx, PLAN['boot'], 1), lateral=boot_ci(gain_lat, PLAN['boot'], 2),
           answer_all_cost_image_only=float(cost_of(o_img['fn'], o_img['fp']).mean()),
           answer_all_cost_with_context=float(cost_of(o_ic['fn'], o_ic['fp']).mean()))
RESULTS['value_of_information'] = VOI
print('Mean realised cost reduction from obtaining clinical context :', {k: round(v, 5) for k, v in VOI['context'].items()})
print('Mean realised cost reduction from obtaining the lateral view :', {k: round(v, 5) for k, v in VOI['lateral'].items()})

# ---------------------------------------------------------------- B. Signal 1: conformal coverage under the primary scheme
rows = []
for k, F in enumerate(FOLDS):
    s = FS[k]; ss = pred_sets(EV['p0'][s], F['q'])
    for j, label in enumerate(LABELS):
        for grp in ['all', 'negative', 'positive']:
            m = EV['mask'][s, j] > .5
            if grp != 'all':
                m &= EV['y'][s, j] == int(grp == 'positive')
            idx = np.flatnonzero(m)
            inc = ss[idx, j, EV['y'][s][idx, j].astype(int)]
            rows.append(dict(fold=k, label=label, group=grp, n=len(idx), covered=int(inc.sum()), set_size=float(ss[idx, j].sum(1).sum())))
cov = pd.DataFrame(rows).groupby(['label', 'group'], as_index=False)[['n', 'covered', 'set_size']].sum()
cov['coverage'] = cov.covered/cov.n.clip(lower=1); cov['mean_set_size'] = cov.set_size/cov.n.clip(lower=1)
cov[['lo', 'hi']] = [wilson(int(c), int(n)) for c, n in zip(cov.covered, cov.n)]
cov['nominal'] = 1-PLAN['conformal_alpha']; cov['covers_nominal'] = cov.hi >= cov.nominal
TABLES['conformal_coverage'] = cov
allc = cov[cov.group == 'all']; posc = cov[(cov.group == 'positive') & (cov.n >= 30)]
RESULTS['conformal'] = dict(labels_all_group_upper_ci_below_nominal=int((allc.hi < allc.nominal).sum()),
                            labels_positive_class_upper_ci_below_nominal=int((posc.hi < posc.nominal).sum()),
                            positive_class_labels_assessed=int(len(posc)), mean_set_size_all=float(allc.mean_set_size.mean()))
display(cov[cov.group != 'negative'].round(3).head(28))
print('Conformal summary:', RESULTS['conformal'])


## 5. Analysis engine
One function runs threshold calibration for the referral gates (Learn then Test and conformal risk control), the budgeted four-action routing (answer, escalate, ask for history, ask for lateral view) and the request comparisons for any missing-history scenario.

In [ ]:
GATES = ['entropy', 'ambiguity', 'sevharm', 'L2D_U', 'L2D_UDMS', 'U', 'UD', 'UDM', 'UDMS', 'oracle']
HARM = {'fn': ('fn0', 'fnc', 'fnl', 'sc_fn_'), 'severe': ('sev0', 'sevc', 'sevl', 'sc_sv_')}
LAM = np.array(PLAN['lam_grid'])
P_ASK = dict(a_c=CTX_A, f_c=CTX_F, a_l=ASK_A, f_l=ASK_F)

def _score(src, gate, kind, harm_key=None):
    if gate == 'oracle':
        h = src[harm_key]
        return h+1e-9*np.random.default_rng(3).random(len(h))
    if gate.startswith('L2D_'):
        return src['sc_l2d_'+gate[4:]]
    return src[HARM[kind][3]+gate] if gate in GATE_SETS else src['sc_'+gate]

def add_holm(df, group_cols):
    df = df.copy(); df['p_holm'] = np.nan
    for _, idx in df.groupby(group_cols).groups.items():
        df.loc[idx, 'p_holm'] = holm(df.loc[idx, 'p'].to_numpy())
    center = 'delta_referral' if 'delta_referral' in df else ('delta' if 'delta' in df else 'delta_workload')
    df['verdict_holm'] = np.where(df.p_holm < 0.05, np.where(df[center] < 0, 'better', 'worse'), 'inconclusive')
    return df

def analyse(scheme, full=False):
    """All guarantee / routing analyses for one deployment scheme. Returns (tables, results)."""
    ev = build_EV(scheme); n = len(ev['case']); fs = [np.flatnonzero(ev['fold'] == k) for k in range(K)]
    cals = [get_cal(k, scheme) for k in range(K)]
    TB, RS = {}, {}
    def ev_slice(k):
        return {key: val[fs[k]] for key, val in ev.items()}
    # ------------------------------------------------ guarantee-calibrated Answer/Escalate gates
    def run_gate(kind, alpha, gate, method):
        h0 = HARM[kind][0]; esc = np.zeros(n, bool); risk_f = []
        for k in range(K):
            t, _ = calibrate_gate(_score(cals[k], gate, kind, h0), cals[k][h0], alpha, PLAN['delta'], method)
            sl = ev_slice(k)
            e_k = _score(sl, gate, kind, h0) >= t
            esc[fs[k]] = e_k; risk_f.append(float((sl[h0]*(~e_k)).mean()))
        loss = ev[h0]*(~esc)
        return dict(esc=esc, risk=float(loss.mean()), referral=float(esc.mean()), fold_risk=risk_f)
    runs, rows = {}, []
    for kind, alphas in PLAN['harm_kinds'].items():
        for alpha in alphas:
            for gate in GATES:
                for method in (['ltt', 'crc'] if gate != 'oracle' else ['ltt']):
                    r = run_gate(kind, alpha, gate, method); runs[(kind, alpha, gate, method)] = r
                    rows.append(dict(harm=kind, alpha=alpha, gate=gate, method=method, referral=r['referral'], realised_risk=r['risk'],
                                     folds_within_target=int(np.sum(np.array(r['fold_risk']) <= alpha)), folds=K,
                                     max_fold_risk=max(r['fold_risk'])))
    GT = pd.DataFrame(rows); TB['gate_at_guaranteed_risk'] = GT
    valid = GT[(GT.method == 'ltt') & (GT.gate != 'oracle')]
    RS['guarantee_internal'] = dict(configurations=int(len(valid)), realised_le_alpha=int((valid.realised_risk <= valid.alpha).sum()),
                                    mean_folds_within_target=float((valid.folds_within_target/valid.folds).mean()),
                                    worst_overshoot=float((valid.realised_risk-valid.alpha).max()))
    contrasts = [('UD', 'U', 'D adds to U'), ('UDM', 'UD', 'M adds to U+D'), ('UDMS', 'UDM', 'S adds to U+D+M'),
                 ('UDMS', 'U', 'Full gate vs binary baseline'), ('UDMS', 'entropy', 'Full gate vs entropy deferral'),
                 ('UDMS', 'ambiguity', 'Full gate vs conformal-ambiguity deferral'),
                 ('UDMS', 'L2D_U', 'Full gate vs learning-to-defer surrogate baseline'),
                 ('UDMS', 'L2D_UDMS', 'Full gate vs learning-to-defer surrogate (all signals)')]
    rows = []
    for kind, alphas in PLAN['harm_kinds'].items():
        for alpha in alphas:
            for a_, b_, name in contrasts:
                ra, rb = runs[(kind, alpha, a_, 'ltt')], runs[(kind, alpha, b_, 'ltt')]
                ci = boot_ci(ra['esc'].astype(float)-rb['esc'].astype(float), PLAN['boot'], 11)
                rows.append(dict(harm=kind, alpha=alpha, contrast=name, referral_a=ra['referral'], referral_b=rb['referral'],
                                 delta_referral=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci), p=ci['p'], mde80=ci['mde80']))
    RQ1 = add_holm(pd.DataFrame(rows), ['contrast']); TB['rq1_referral_contrasts'] = RQ1
    # ------------------------------------------------ budgeted routing with up to four actions
    C0 = ev['fn0']+FPW*ev['fp0']; CC = ev['fnc']+FPW*ev['fpc']; CL = ev['fnl']+FPW*ev['fpl']
    def acts_for(name, k, budget, p):
        sl = ev_slice(k)
        if name == 'random':
            return topk_escalate(np.random.default_rng(100+k).random(len(fs[k])), budget)
        if name == 'binary_L2D':
            return topk_escalate(sl['sc_l2d_UDMS'], budget)
        if name == 'binary_entropy':
            return topk_escalate(sl['sc_entropy'], budget)
        if name == 'oracle_binary':
            return topk_escalate((sl['fn0']+FPW*sl['fp0']), budget)
        F = 'U' if name == 'binary_U' else 'UDMS'
        allow = {'binary_U': (False, False), 'binary_UDMS': (False, False), 'ask_ctx_UDMS': (True, False),
                 'ask_lat_UDMS': (False, True), 'ask_both_UDMS': (True, True)}[name]
        pp = dict(p, allow_ctx=allow[0], allow_lat=allow[1])
        return pick_lambda4(sl['c0hat_'+F], sl['gc'], sl['gl'], sl['ctx_ok'], sl['lat_ok'], EXPERT_E, pp, budget, LAM)[1]
    def evaluate(name, budget, p=P_ASK):
        act = np.zeros(n, int)
        for k in range(K):
            act[fs[k]] = acts_for(name, k, budget, p)
        return dict(act=act, workload=workload4(act, p), cost=realised4(act, C0, CC, CL, EXPERT_E, p),
                    fn=realised4(act, ev['fn0'], ev['fnc'], ev['fnl'], EXPERT_E, p),
                    fp=realised4(act, ev['fp0'], ev['fpc'], ev['fpl'], EXPERT_E, p),
                    severe=realised4(act, ev['sev0'], ev['sevc'], ev['sevl'], EXPERT_E, p))
    POLS = ['random', 'binary_entropy', 'binary_L2D', 'binary_U', 'binary_UDMS', 'ask_ctx_UDMS', 'ask_lat_UDMS', 'ask_both_UDMS', 'oracle_binary']
    SIMR = {}; rows = []
    for b in PLAN['budgets']:
        for name in POLS:
            r = evaluate(name, b); SIMR[(b, name)] = r
            rows.append(dict(budget=b, policy=name, workload=r['workload'].mean(), escalate=(r['act'] == 1).mean(),
                             ask_context=(r['act'] == 2).mean(), ask_lateral=(r['act'] == 3).mean(), cost=r['cost'].mean(),
                             fn=r['fn'].mean(), fp=r['fp'].mean(), severe_miss=r['severe'].mean()))
    FR = pd.DataFrame(rows); TB['budget_frontier'] = FR
    pairs = [('ask_ctx_UDMS', 'binary_UDMS', 'RQ2a: add ASK-CONTEXT to the full gate'),
             ('ask_lat_UDMS', 'binary_UDMS', 'RQ2b: add ASK-LATERAL to the full gate'),
             ('ask_both_UDMS', 'binary_UDMS', 'RQ2: add both Ask actions to the full gate'),
             ('binary_UDMS', 'binary_U', 'RQ1: full signals vs uncertainty only (binary)'),
             ('binary_UDMS', 'binary_entropy', 'Full gate vs entropy deferral (binary)'),
             ('binary_UDMS', 'binary_L2D', 'Full gate vs learning-to-defer surrogate baseline (binary)'),
             ('ask_both_UDMS', 'binary_U', 'Whole system vs binary defer-or-act baseline'),
             ('binary_UDMS', 'random', 'Sanity: full gate vs random escalation')]
    rows = []
    for b in PLAN['budgets']:
        for pa, pb, name in pairs:
            for metric in ['cost', 'fn', 'severe']:
                ci = boot_ci(SIMR[(b, pa)][metric]-SIMR[(b, pb)][metric], PLAN['boot'], 21)
                rows.append(dict(budget=b, contrast=name, policy_a=pa, policy_b=pb, metric=metric, delta=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci), p=ci['p'], mde80=ci['mde80']))
    FC = add_holm(pd.DataFrame(rows), ['contrast', 'metric']); TB['budget_contrasts'] = FC
    # cost-unaware "ask whenever context is missing" (BODHI-style proxy), compared at MATCHED workload
    act_all = np.where(ev['ctx_ok'], 2, 0); wl_all = workload4(act_all, P_ASK)
    bm = float(wl_all.mean())
    ask_all = dict(cost=realised4(act_all, C0, CC, CL, EXPERT_E, P_ASK), workload=wl_all)
    match = {name: evaluate(name, bm) for name in ['binary_UDMS', 'ask_both_UDMS']}
    rows = []
    for name, r in match.items():
        ci = boot_ci(r['cost']-ask_all['cost'], PLAN['boot'], 22)
        rows.append(dict(reference='ask whenever context missing (cost-unaware)', matched_workload=bm, policy=name,
                         delta_cost_vs_reference=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci)))
    BOD = pd.DataFrame(rows); TB['bodhi_proxy_matched_workload'] = BOD
    # ------------------------------------------------ guarantee with Ask actions (Learn-then-Test; losses need not be monotone)
    def family_guarantee(kind, alpha, allow_ctx, allow_lat, p=P_ASK):
        h0, hc, hl, _ = HARM[kind]; pp = dict(p, allow_ctx=allow_ctx, allow_lat=allow_lat)
        wl = np.zeros(n); asked_c = np.zeros(n); asked_l = np.zeros(n); risk = np.zeros(n); chosen = []
        for k in range(K):
            cl_ = cals[k]
            A = np.stack([route4(cl_['c0hat_UDMS'], cl_['gc'], cl_['gl'], cl_['ctx_ok'], cl_['lat_ok'], lam, EXPERT_E, pp) for lam in LAM], axis=1)
            L = np.stack([realised4(A[:, g], cl_[h0], cl_[hc], cl_[hl], 0., pp) for g in range(len(LAM))], axis=1)
            idx = ltt_index(L, alpha, PLAN['delta']); chosen.append(idx); sl = ev_slice(k)
            act = np.ones(len(fs[k]), int) if idx < 0 else route4(sl['c0hat_UDMS'], sl['gc'], sl['gl'], sl['ctx_ok'], sl['lat_ok'], LAM[idx], EXPERT_E, pp)
            wl[fs[k]] = workload4(act, pp); asked_c[fs[k]] = act == 2; asked_l[fs[k]] = act == 3
            risk[fs[k]] = realised4(act, sl[h0], sl[hc], sl[hl], 0., pp)
        return dict(workload=wl, ask_c=asked_c, ask_l=asked_l, risk=risk, idx=chosen)
    rows = []
    for kind, alphas in PLAN['harm_kinds'].items():
        for alpha in alphas:
            r2 = family_guarantee(kind, alpha, False, False)
            for label, ac, al in [('ask_context', True, False), ('ask_lateral', False, True), ('ask_both', True, True)]:
                r3 = family_guarantee(kind, alpha, ac, al)
                ci = boot_ci(r3['workload']-r2['workload'], PLAN['boot'], 41)
                rows.append(dict(harm=kind, alpha=alpha, variant=label, workload_binary=float(r2['workload'].mean()),
                                 workload_with_ask=float(r3['workload'].mean()), delta_workload=ci['mean'], lo=ci['lo'], hi=ci['hi'],
                                 verdict=verdict(ci), p=ci['p'], mde80=ci['mde80'], ask_context_rate=float(r3['ask_c'].mean()), ask_lateral_rate=float(r3['ask_l'].mean()),
                                 risk_binary=float(r2['risk'].mean()), risk_with_ask=float(r3['risk'].mean()),
                                 no_valid_lambda_folds=int(sum(i < 0 for i in r3['idx']))))
    G3 = add_holm(pd.DataFrame(rows), ['variant']); TB['ask_at_guaranteed_risk'] = G3
    # ------------------------------------------------ "ask for the missing context first, then decide" (two-stage workflow)
    # Context is requested for EVERY patient lacking it; the gate then runs on the completed information (retrieval can fail).
    TS = {}
    if scheme != 'natural' and 'natural' in PLAN['schemes']:
        nat = build_EV('natural'); nat_cals = [get_cal(k, 'natural') for k in range(K)]
        fail = hash_uniform(ev['case'], 'askfail') < CTX_F; got = ev['ctx_ok'] & ~fail
        ask_wl = np.where(ev['ctx_ok'], CTX_A+CTX_F, 0.)
        comp = lambda key: np.where(got, nat[key], ev[key])
        c0h = comp('c0hat_UDMS'); fn_c, fp_c, sev_c = comp('fn0'), comp('fp0'), comp('sev0'); cost_c = fn_c+FPW*fp_c
        rows = []
        for b in PLAN['budgets']:
            rem = b-ask_wl.mean()
            if rem <= 0:
                continue
            act = np.zeros(n, int); z = lambda m_: np.zeros(m_)
            for k in range(K):
                s_ = fs[k]
                act[s_] = pick_lambda4(c0h[s_], z(len(s_)), z(len(s_)), np.zeros(len(s_), bool), np.zeros(len(s_), bool), EXPERT_E,
                                       dict(P_ASK, allow_ctx=False, allow_lat=False), rem, LAM)[1]
            esc_ = act == 1
            for metric, base_, comp_ in [('cost', SIMR[(b, 'binary_UDMS')]['cost'], np.where(esc_, EXPERT_E*cost_c, cost_c)),
                                         ('fn', SIMR[(b, 'binary_UDMS')]['fn'], np.where(esc_, EXPERT_E*fn_c, fn_c)),
                                         ('severe', SIMR[(b, 'binary_UDMS')]['severe'], np.where(esc_, EXPERT_E*sev_c, sev_c))]:
                ci = boot_ci(comp_-base_, PLAN['boot'], 61)
                rows.append(dict(budget=b, metric=metric, escalate=float(esc_.mean()), ask=float(ask_wl.mean()), delta_vs_binary_UDMS=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci)))
        TS['budget'] = pd.DataFrame(rows)
        rows = []
        for kind, alphas in PLAN['harm_kinds'].items():
            h0 = HARM[kind][0]
            for alpha in alphas:
                esc_b = runs[(kind, alpha, 'UDMS', 'ltt')]['esc']; esc2 = np.zeros(n, bool); risk2 = np.zeros(n)
                for k in range(K):
                    cal = cals[k]; calN = nat_cals[k]
                    failc = hash_uniform(FOLDS[k]['cal_case'], 'askfail') < CTX_F; gotc = cal['ctx_ok'] & ~failc
                    sc_cal = np.where(gotc, calN[HARM[kind][3]+'UDMS'], cal[HARM[kind][3]+'UDMS']); h_cal = np.where(gotc, calN[h0], cal[h0])
                    t, _ = calibrate_gate(sc_cal, h_cal, alpha, PLAN['delta'], 'ltt'); s_ = fs[k]
                    sc_ev = np.where(got[s_], nat[HARM[kind][3]+'UDMS'][s_], ev[HARM[kind][3]+'UDMS'][s_])
                    h_ev = np.where(got[s_], nat[h0][s_], ev[h0][s_])
                    esc2[s_] = sc_ev >= t; risk2[s_] = h_ev*(~esc2[s_])
                wl_b = esc_b.astype(float); wl_2 = esc2.astype(float)+ask_wl
                ci = boot_ci(wl_2-wl_b, PLAN['boot'], 62)
                saving = float(wl_b.mean()-esc2.mean())
                rows.append(dict(harm=kind, alpha=alpha, workload_binary=float(wl_b.mean()), workload_ask_then_gate=float(wl_2.mean()),
                                 escalations_saved=saving, break_even_request_workload=max(saving/max(float(ev['ctx_ok'].mean()), 1e-9)-CTX_F, 0.0),
                                 delta_workload=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci), risk_ask_then_gate=float(risk2.mean())))
        TS['guarantee'] = pd.DataFrame(rows)
        TB['ask_missing_then_gate_budget'] = TS['budget']; TB['ask_missing_then_gate_guarantee'] = TS['guarantee']
        RS['ask_missing_then_gate'] = dict(budget=TS['budget'][TS['budget'].metric == 'cost'][['budget', 'delta_vs_binary_UDMS', 'lo', 'hi', 'verdict']].to_dict('records'),
                                           guarantee=TS['guarantee'][['harm', 'alpha', 'delta_workload', 'lo', 'hi', 'verdict', 'break_even_request_workload']].to_dict('records'))
    # ------------------------------------------------ summary used by the scorecard
    prim = PLAN['primary_budget']
    def pick(df, **kw):
        m = np.ones(len(df), bool)
        for k_, v_ in kw.items():
            m &= (df[k_] == v_).to_numpy()
        return df[m]
    RS['rq1_full_vs_binary_referral'] = pick(RQ1, contrast='Full gate vs binary baseline')[['harm', 'alpha', 'delta_referral', 'lo', 'hi', 'verdict']].to_dict('records')
    RS['ask_primary_budget'] = {name: pick(FC, budget=prim, policy_a=pa, policy_b='binary_UDMS', metric='cost')[['delta', 'lo', 'hi', 'verdict']].to_dict('records')[0]
                                for name, pa in [('context', 'ask_ctx_UDMS'), ('lateral', 'ask_lat_UDMS'), ('both', 'ask_both_UDMS')]}
    pf = pick(FR, budget=prim, policy='ask_both_UDMS').iloc[0]
    RS['ask_rates_primary'] = dict(context=float(pf.ask_context), lateral=float(pf.ask_lateral))
    RS['rq1_budget_cost'] = pick(FC, policy_a='binary_UDMS', policy_b='binary_U', metric='cost')[['budget', 'delta', 'lo', 'hi', 'verdict']].to_dict('records')
    RS['bodhi_proxy'] = BOD.to_dict('records')
    RS['ask_at_guarantee'] = G3[['harm', 'alpha', 'variant', 'delta_workload', 'lo', 'hi', 'verdict']].to_dict('records')
    ctxg = pick(G3, variant='ask_context')
    RS['ask_context_guarantee_counts'] = {k_: int(v_) for k_, v_ in ctxg.verdict.value_counts().items()}
    if full:
        # ---- Ask economics: when does each request pay off?
        rows = []
        for a in PLAN['ctx_grid_a']:
            for f in PLAN['ctx_grid_f']:
                for b in [0.2, 0.3]:
                    p = dict(P_ASK, a_c=a, f_c=f); s3 = evaluate('ask_ctx_UDMS', b, p); s2 = SIMR[(b, 'binary_UDMS')]
                    ci = boot_ci(s3['cost']-s2['cost'], PLAN['boot'], 31)
                    rows.append(dict(request='context', a=a, f=f, budget=b, ask_rate=float((s3['act'] == 2).mean()), delta_cost=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci)))
        for a in PLAN['lat_grid_a']:
            for b in [0.2, 0.3]:
                p = dict(P_ASK, a_l=a); s3 = evaluate('ask_lat_UDMS', b, p); s2 = SIMR[(b, 'binary_UDMS')]
                ci = boot_ci(s3['cost']-s2['cost'], PLAN['boot'], 32)
                rows.append(dict(request='lateral', a=a, f=ASK_F, budget=b, ask_rate=float((s3['act'] == 3).mean()), delta_cost=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci)))
        ECON = pd.DataFrame(rows); TB['ask_break_even_grid'] = ECON
        RS['ask_break_even'] = {r_: dict(cells=int((ECON.request == r_).sum()), ask_better=int(((ECON.request == r_) & (ECON.verdict == 'better')).sum()),
                                         ask_worse=int(((ECON.request == r_) & (ECON.verdict == 'worse')).sum())) for r_ in ['context', 'lateral']}
    return TB, RS, dict(ev=ev, fs=fs, cals=cals, SIMR=SIMR, runs=runs, C0=C0, evaluate=evaluate)
print('Analysis engine ready.')


## 6. Primary results and robustness across missing-history scenarios
Primary scenario: 30% of histories withheld at random. Covers referral at each risk target, budgeted routing and requests, the matched-workload comparison with requesting every missing history, break-even grids, the mismatch and missing-history stress tests, and the alternative scenarios.

In [ ]:
TB, RS, CTX = analyse(PRIMARY, full=True)
TABLES.update(TB); RESULTS.update(RS)
GT, RQ1, FR, FC, G3, ECON, BOD = (TB['gate_at_guaranteed_risk'], TB['rq1_referral_contrasts'], TB['budget_frontier'],
                                  TB['budget_contrasts'], TB['ask_at_guaranteed_risk'], TB['ask_break_even_grid'], TB['bodhi_proxy_matched_workload'])
SIMR, runs = CTX['SIMR'], CTX['runs']
prim = PLAN['primary_budget']
print('=== Guarantee-calibrated Answer/Escalate gates: fraction of patients that must be escalated (lower = less clinician workload)')
display(GT[GT.method == 'ltt'].pivot_table(index=['harm', 'alpha'], columns='gate', values='referral').round(3))
print('Guarantee check (LTT, delta=%.2f):' % PLAN['delta'], RS['guarantee_internal'])
print('\n=== RQ1: does adding signals reduce the workload needed at the SAME guaranteed risk?')
display(RQ1.round(4))
# threshold-free ranking quality for the critical-finding event
sev = EV['sev0'] > .5; rows = []
for a_, b_, name in [('U', 'UD', 'D adds to U'), ('UD', 'UDM', 'M adds to U+D'), ('UDM', 'UDMS', 'S adds to U+D+M'), ('U', 'UDMS', 'Full vs U')]:
    r = auc_delta_ci(sev, EV['sc_sv_'+a_], EV['sc_sv_'+b_], reps=min(PLAN['boot'], 600), seed=5)
    for metric in ['auroc', 'auprc']:
        rows.append(dict(contrast=name, metric=metric, **r[metric]))
AUCT = pd.DataFrame(rows); TABLES['rq1_ranking_auc'] = AUCT; display(AUCT.round(4))
RESULTS['rq1_ranking'] = AUCT[AUCT.contrast == 'Full vs U'][['metric', 'a', 'b', 'delta', 'lo', 'hi']].to_dict('records')

print('\n=== Budgeted routing (workload fixed by design); primary budget =', prim)
display(FR[FR.budget == prim].round(4))
display(FC[(FC.budget == prim)].round(5))
print('\n=== Cost-unaware "ask whenever the context is missing" vs the cost-aware gates at the SAME workload')
display(BOD.round(5))
print('\n=== "Ask for the missing context first, then decide" (two-stage workflow) vs the binary gate')
display(TB['ask_missing_then_gate_budget'].round(5)); display(TB['ask_missing_then_gate_guarantee'].round(4))
print('break_even_request_workload = largest workload cost of one context request for which asking first would still pay off')
print('\n=== Ask actions under the finite-sample guarantee')
display(G3.round(4))
print('\n=== When does each information request pay off? (negative delta_cost = Ask helps)')
display(ECON.pivot_table(index=['request', 'a'], columns=['budget', 'f'], values='delta_cost').round(5))
print('Break-even summary:', RESULTS['ask_break_even'])

# ---------------------------------------------------------------- Signal 2 stress test (mismatched context) and Signal 3 stress test (context withheld)
rows = []
for k, F in enumerate(FOLDS):
    sc_m = {g: F['ev_%s_sc_fn_%s' % (PRIMARY, g)] for g in ['U', 'UD', 'UDM', 'UDMS']}
    thr = {g: np.quantile(sc_m[g], .7) for g in sc_m}
    for g in thr:
        rows.append(dict(gate=g, matched_escalation=float((sc_m[g] >= thr[g]).mean()), mismatched_escalation=float((F['mis_sc_fn_'+g] >= thr[g]).mean())))
mis_esc = pd.DataFrame(rows).groupby('gate', as_index=False).mean(numeric_only=True)
availv = EV['avail']; both = np.concatenate([F['mis_avail'] for F in FOLDS]) & availv
mis_d = np.concatenate([F['mis_dmean'] for F in FOLDS])
y_det = np.r_[np.zeros(both.sum()), np.ones(both.sum())]
det = {'D signal (mean JS divergence)': float(roc_auc_score(y_det, np.r_[EV['dmean'][both], mis_d[both]])),
       'U-only gate score': float(roc_auc_score(y_det, np.r_[EV['sc_fn_U'][both], np.concatenate([F['mis_sc_fn_U'] for F in FOLDS])[both]])),
       'Full gate score': float(roc_auc_score(y_det, np.r_[EV['sc_fn_UDMS'][both], np.concatenate([F['mis_sc_fn_UDMS'] for F in FOLDS])[both]]))}
fn_mis = np.concatenate([F['mis_fn0'] for F in FOLDS])
RESULTS['stress_mismatch'] = dict(detection_auroc=det, matched_fn=float(EV['fn0'][both].mean()), mismatched_fn=float(fn_mis[both].mean()),
                                  escalation_at_matched_70pct_threshold=mis_esc.round(4).to_dict('records'))
TABLES['stress_mismatch_escalation'] = mis_esc
print('\nSignal 2 (mismatched context) stress test:', json.dumps(RESULTS['stress_mismatch'], indent=1))
# Signal 3: context withheld. Compare the three schemes; 'struct40' is a masking scheme the gate models never saw in training.
rows = []
for scheme in PLAN['schemes']:
    e_s = build_EV(scheme)
    s_ = e_s['ctx_ok']; a_ = e_s['avail']
    rows.append(dict(scheme=scheme, context_available=float(a_.mean()), answer_all_fn=float(e_s['fn0'].mean()),
                     answer_all_cost=float((e_s['fn0']+FPW*e_s['fp0']).mean()),
                     ask_context_would_help=float(((e_s['fn0']+FPW*e_s['fp0'])-(e_s['fnc']+FPW*e_s['fpc']))[s_].mean()) if s_.any() else np.nan,
                     predicted_gain_corr=float(np.corrcoef(e_s['gc'][s_], ((e_s['fn0']+FPW*e_s['fp0'])-(e_s['fnc']+FPW*e_s['fpc']))[s_])[0, 1]) if s_.sum() > 30 else np.nan))
SIG3 = pd.DataFrame(rows); TABLES['signal3_context_withheld'] = SIG3; display(SIG3.round(4))
RESULTS['signal3'] = SIG3.to_dict('records')

# ---------------------------------------------------------------- robustness across deployment schemes (primary analysed above)
rows = []
for scheme in PLAN['schemes']:
    tb, rs = (TB, RS) if scheme == PRIMARY else analyse(scheme, False)[:2]
    if scheme != PRIMARY:
        TABLES[f'scheme_{scheme}_rq1'] = tb['rq1_referral_contrasts']; TABLES[f'scheme_{scheme}_ask_guarantee'] = tb['ask_at_guaranteed_risk']
    rq = tb['rq1_referral_contrasts']; rq = rq[rq.contrast == 'Full gate vs binary baseline']
    g3 = tb['ask_at_guaranteed_risk']
    rows.append(dict(scheme=scheme, guarantee_ok=f"{rs['guarantee_internal']['realised_le_alpha']}/{rs['guarantee_internal']['configurations']}",
                     rq1_full_vs_binary={k_: int(v_) for k_, v_ in rq.verdict.value_counts().items()},
                     ask_context_budget_cost=f"{rs['ask_primary_budget']['context']['delta']:+.5f} [{rs['ask_primary_budget']['context']['lo']:+.5f},{rs['ask_primary_budget']['context']['hi']:+.5f}]",
                     ask_context_verdict=rs['ask_primary_budget']['context']['verdict'],
                     ask_both_verdict=rs['ask_primary_budget']['both']['verdict'],
                     ask_context_guarantee_workload={k_: int(v_) for k_, v_ in g3[g3.variant == 'ask_context'].verdict.value_counts().items()},
                     ask_context_rate=rs['ask_rates_primary']['context']))
ROB = pd.DataFrame(rows); TABLES['scheme_robustness'] = ROB; RESULTS['scheme_robustness'] = ROB.astype(str).to_dict('records')
pd.set_option('display.max_colwidth', 80); display(ROB)


# ---------------------------------------------------------------- clinical-unit reporting (per 1,000 patients)
rows = []
for r in RQ1[RQ1.contrast.isin(['Full gate vs binary baseline', 'Full gate vs entropy deferral', 'Full gate vs conformal-ambiguity deferral',
                                'Full gate vs learning-to-defer surrogate baseline'])].itertuples():
    rows.append(dict(harm=r.harm, alpha=r.alpha, comparison=r.contrast,
                     escalations_avoided_per_1000=-1000*r.delta_referral, lo=-1000*r.hi, hi=-1000*r.lo, p_holm=r.p_holm, verdict_holm=r.verdict_holm,
                     minimum_detectable_per_1000=1000*r.mde80))
PER1000 = pd.DataFrame(rows); TABLES['per_1000_patients'] = PER1000
print('\n=== Clinician escalations avoided per 1,000 patients at the SAME guaranteed risk (positive = full gate needs fewer)')
display(PER1000[PER1000.comparison.str.contains('binary|entropy|learning')].round(2))
RESULTS['per_1000'] = PER1000.to_dict('records')

# ---------------------------------------------------------------- net-benefit (decision-curve) view of the routing policies
evaluate = CTX['evaluate']; FINE = [round(x, 2) for x in np.arange(0.05, 0.55, 0.05)]
lam_grid = np.linspace(0, 0.25, 51); C0 = CTX['C0']; base_cost = float(C0.mean()); rows = []
for name in ['random', 'binary_entropy', 'binary_L2D', 'binary_U', 'binary_UDMS', 'ask_both_UDMS', 'oracle_binary']:
    pts = [(0., 0.)]+[(float(base_cost-evaluate(name, b)['cost'].mean()), float(evaluate(name, b)['workload'].mean())) for b in FINE]
    for lam in lam_grid:
        rows.append(dict(policy=name, price_of_workload=float(lam), net_benefit=max(g-lam*w for g, w in pts)))
DCV = pd.DataFrame(rows); DCV['price_of_workload'] = DCV.price_of_workload.round(3); TABLES['decision_curve'] = DCV
print('Net benefit = cost avoided minus (price x workload); the best operating budget is chosen per price. Higher is better.')
display(DCV[DCV.price_of_workload.isin([0.02, 0.05, 0.10, 0.15])].pivot_table(index='price_of_workload', columns='policy', values='net_benefit').round(5))


## 6b. Stability across fold assignments
Repeats the cross-fitting with independent patient-to-fold assignments. Set `seeds=[0]` in the constants cell to skip.

In [ ]:
# ---------------------------------------------------------------- stability: independent patient-fold assignments (seeds)
# Everything above used fold assignment 0. Here the whole cross-fitting is repeated with other assignments of patients to folds and the
# headline numbers are compared. Set PLAN['seeds'] = [0] to skip (saves roughly 15-45 minutes on a free CPU runtime).
PE = PLAN['primary_endpoint']
def seed_summary(tb, rs, s):
    r = tb['rq1_referral_contrasts']; r = r[(r.harm == PE['harm']) & (r.alpha == PE['alpha'])]
    get = lambda name: float(r[r.contrast == name].delta_referral.iloc[0])
    g = rs['guarantee_internal']
    return dict(seed=s, guarantee=f"{g['realised_le_alpha']}/{g['configurations']}", worst_overshoot=g['worst_overshoot'],
                full_vs_binary=get('Full gate vs binary baseline'), full_vs_entropy=get('Full gate vs entropy deferral'),
                full_vs_l2d=get('Full gate vs learning-to-defer surrogate baseline'),
                ask_context_cost_delta=rs['ask_primary_budget']['context']['delta'], ask_both_cost_delta=rs['ask_primary_budget']['both']['delta'])
SEED_ROWS = [seed_summary(TB, RS, 0)]
for s_ in PLAN['seeds'][1:]:
    run_seed(s_); load_seed()
    tb_, rs_, _ = analyse(PRIMARY, False); SEED_ROWS.append(seed_summary(tb_, rs_, s_))
run_seed(0); load_seed()                       # restore the primary assignment for every later cell
SEEDS = pd.DataFrame(SEED_ROWS); TABLES['seed_stability'] = SEEDS; display(SEEDS.round(5))
num = SEEDS.drop(columns=['seed', 'guarantee']).astype(float)
RESULTS['seed_stability'] = dict(seeds=len(SEEDS), mean=num.mean().round(5).to_dict(),
                                 sd=num.std(ddof=1).round(5).to_dict() if len(SEEDS) > 1 else {},
                                 same_sign_full_vs_binary=bool((np.sign(SEEDS.full_vs_binary) == np.sign(SEEDS.full_vs_binary.iloc[0])).all()))
print('Across', len(SEEDS), 'fold assignments:', RESULTS['seed_stability'])


## 6c. Simultaneous targets for each critical finding
One target per finding, tested simultaneously. Compares the union of finding-specific gates with one shared score and a single threshold.

In [ ]:
# ---------------------------------------------------------------- per-critical-finding risk guarantees (simultaneous)
# A pooled burden hides which finding is missed. Here each critical finding (severity >= 0.7) gets its OWN target on
#   risk_j = E[ 1{finding j present and missed by the classifier} x 1{case answered automatically} ]   (per patient)
# set to a fraction of that finding's answer-everything miss rate. Two valid constructions, both with joint confidence 1-delta
# (Bonferroni split delta/#findings):
#   (A) union of per-finding gates: escalate if ANY finding-specific gate fires (each gate calibrated on its own risk);
#   (B) one shared score and ONE threshold, accepted only if every finding's risk passes (multi-risk Learn-then-Test).
CRIT = PLAN['crit_labels']; nC = len(CRIT)
def critical_analysis(scheme):
    ev = build_EV(scheme); n = len(ev['case']); fs = [np.flatnonzero(ev['fold'] == k) for k in range(K)]
    cals = [get_cal(k, scheme) for k in range(K)]
    base = {lab: float(ev['miss_'+lab].mean()) for lab in CRIT}
    events_cal = {lab: float(np.mean([c['miss_'+lab].sum() for c in cals])) for lab in CRIT}
    rows, escs = [], {}
    for frac in PLAN['crit_fracs']:
        alph = {lab: frac*base[lab] for lab in CRIT}
        for F in ['U', 'UDMS']:
            esc_u = np.zeros(n, bool); esc_s = np.zeros(n, bool); fb_u = 0; fb_s = 0
            for k in range(K):
                s_ = fs[k]; cal = cals[k]; e_u = np.zeros(len(s_), bool)
                for lab in CRIT:
                    t, idx = calibrate_gate(cal[f'scj_{lab}_{F}'], cal['miss_'+lab], alph[lab], PLAN['delta']/nC, 'ltt')
                    fb_u += int(idx < 0); e_u |= ev[f'scj_{lab}_{F}'][s_] >= t
                esc_u[s_] = e_u
                sc_cal = np.max([cal[f'scj_{lab}_{F}']/max(base[lab], 1e-9) for lab in CRIT], axis=0)
                thr = gate_thresholds(sc_cal)
                Ls = [gate_loss_matrix(sc_cal, cal['miss_'+lab], thr) for lab in CRIT]
                idx = multi_ltt_index(Ls, [alph[l] for l in CRIT], PLAN['delta']/nC); fb_s += int(idx < 0)
                t = -np.inf if idx < 0 else thr[idx]
                sc_ev = np.max([ev[f'scj_{lab}_{F}'][s_]/max(base[lab], 1e-9) for lab in CRIT], axis=0)
                esc_s[s_] = sc_ev >= t
            for name, esc, fb in [('union_of_per_finding_gates', esc_u, fb_u), ('shared_score_single_threshold', esc_s, fb_s)]:
                escs[(frac, F, name)] = esc
                rec = dict(target_fraction=frac, gate=F, method=name, referral=float(esc.mean()), folds_with_no_valid_threshold=fb, folds=K*(nC if name.startswith('union') else 1))
                ok = True
                for lab in CRIT:
                    r = float((ev['miss_'+lab]*(~esc)).mean()); rec[f'risk_{lab}'] = r; rec[f'target_{lab}'] = alph[lab]; ok &= r <= alph[lab]
                rec['all_targets_met'] = bool(ok); rows.append(rec)
    T = pd.DataFrame(rows)
    rows = []
    for frac in PLAN['crit_fracs']:
        for name in ['union_of_per_finding_gates', 'shared_score_single_threshold']:
            ci = boot_ci(escs[(frac, 'UDMS', name)].astype(float)-escs[(frac, 'U', name)].astype(float), PLAN['boot'], 81)
            rows.append(dict(target_fraction=frac, method=name, delta_referral_UDMS_vs_U=ci['mean'], lo=ci['lo'], hi=ci['hi'], p=ci['p'], mde80=ci['mde80']))
    C = pd.DataFrame(rows); C['p_holm'] = holm(C.p.to_numpy())
    C['verdict_holm'] = np.where(C.p_holm < .05, np.where(C.delta_referral_UDMS_vs_U < 0, 'better', 'worse'), 'inconclusive')
    cmp = []
    for frac in PLAN['crit_fracs']:
        ci = boot_ci(escs[(frac, 'UDMS', 'union_of_per_finding_gates')].astype(float)-escs[(frac, 'UDMS', 'shared_score_single_threshold')].astype(float), PLAN['boot'], 82)
        cmp.append(dict(target_fraction=frac, comparison='union of per-finding gates minus shared single threshold (full gate)', delta_referral=ci['mean'], lo=ci['lo'], hi=ci['hi']))
    return T, C, pd.DataFrame(cmp), base, events_cal

CRIT_T, CRIT_C, CRIT_M, CRIT_BASE, CRIT_EVENTS = critical_analysis(PRIMARY)
TABLES['critical_finding_guarantee'] = CRIT_T; TABLES['critical_finding_contrast'] = CRIT_C; TABLES['critical_finding_method_comparison'] = CRIT_M
print('Answer-everything miss rate per critical finding (per patient):', {k: round(v, 4) for k, v in CRIT_BASE.items()})
print('Average number of miss events per calibration fold (what limits per-finding guarantees):', {k: round(v, 1) for k, v in CRIT_EVENTS.items()})
display(CRIT_T.round(4)); display(CRIT_C.round(4)); display(CRIT_M.round(4))
RESULTS['critical_findings'] = dict(base_miss_rate=CRIT_BASE, cal_events_per_fold=CRIT_EVENTS,
    union_all_targets_met=int(CRIT_T[CRIT_T.method == 'union_of_per_finding_gates'].all_targets_met.sum()), union_configs=int((CRIT_T.method == 'union_of_per_finding_gates').sum()),
    shared_all_targets_met=int(CRIT_T[CRIT_T.method != 'union_of_per_finding_gates'].all_targets_met.sum()), shared_configs=int((CRIT_T.method != 'union_of_per_finding_gates').sum()),
    table=CRIT_T.to_dict('records'), full_vs_U=CRIT_C.to_dict('records'), method_comparison=CRIT_M.to_dict('records'))
print({k: v for k, v in RESULTS['critical_findings'].items() if k.endswith('met') or k.endswith('configs')})


## 6d. Bootstrap-ensemble spread (first fold assignment)
Tests whether spread across bootstrap-fitted heads changes referrals at the same risk target.

In [ ]:
# ---------------------------------------------------------------- bootstrap-ensemble disagreement signal E (seed-0 experiment)
# E = spread of calibrated predictions across heads trained on patient-level bootstrap resamples (epistemic uncertainty, not just the entropy of
# one model). Question: do U+E and U+D+M+S+E need fewer escalations than U and U+D+M+S at the same guaranteed risk?
def ensemble_signal_test():
    key = lambda role, name: f'{role}_{PRIMARY}_{name}'
    if key('ev', 'sc_fn_UE') not in FOLDS[0]:
        raise KeyError('ensemble scores were not computed for this fold assignment (PLAN[ensemble_seeds]).')
    ev = {g: np.concatenate([F[key('ev', 'sc_fn_'+g)] for F in FOLDS]) for g in ['U', 'UE', 'UDMS', 'UDMSE']}
    ev.update({'sv_'+g: np.concatenate([F[key('ev', 'sc_sv_'+g)] for F in FOLDS]) for g in ['U', 'UE', 'UDMS', 'UDMSE']})
    ev['fn0'] = np.concatenate([F[key('ev', 'fn0')] for F in FOLDS]); ev['sev0'] = np.concatenate([F[key('ev', 'sev0')] for F in FOLDS])
    rows, escs = [], {}
    for kind, alphas in PLAN['harm_kinds'].items():
        pre = 'sc_fn_' if kind == 'fn' else 'sc_sv_'; h = 'fn0' if kind == 'fn' else 'sev0'
        for alpha in alphas:
            for g in ['U', 'UE', 'UDMS', 'UDMSE']:
                esc = np.zeros(NE, bool)
                for k in range(K):
                    t, _ = calibrate_gate(FOLDS[k][key('cal', pre+g)], FOLDS[k][key('cal', h)], alpha, PLAN['delta'], 'ltt')
                    s_ = FS[k]; esc[s_] = (FOLDS[k][key('ev', pre+g)]) >= t
                escs[(kind, alpha, g)] = esc
                rows.append(dict(harm=kind, alpha=alpha, gate=g, referral=float(esc.mean()), realised_risk=float((ev[h]*(~esc)).mean())))
    G = pd.DataFrame(rows); TABLES['ensemble_signal_gates'] = G
    cons = []
    for kind, alphas in PLAN['harm_kinds'].items():
        for alpha in alphas:
            for a_, b_, name in [('UE', 'U', 'E adds to U'), ('UDMSE', 'UDMS', 'E adds to U+D+M+S'), ('UDMSE', 'U', 'All signals incl. E vs U')]:
                ci = boot_ci(escs[(kind, alpha, a_)].astype(float)-escs[(kind, alpha, b_)].astype(float), PLAN['boot'], 91)
                cons.append(dict(harm=kind, alpha=alpha, contrast=name, delta_referral=ci['mean'], lo=ci['lo'], hi=ci['hi'], p=ci['p'], mde80=ci['mde80']))
    C = pd.DataFrame(cons); C['p_holm'] = holm(C.p.to_numpy()); C['verdict_holm'] = np.where(C.p_holm < .05, np.where(C.delta_referral < 0, 'better', 'worse'), 'inconclusive')
    TABLES['ensemble_signal_contrasts'] = C
    sevy = ev['sev0'] > .5; r = auc_delta_ci(sevy, ev['sv_U'], ev['sv_UE'], reps=min(PLAN['boot'], 600), seed=95)
    RESULTS['ensemble_signal'] = dict(contrasts=C.to_dict('records'), auroc_critical_miss=dict(U=r['auroc']['a'], UE=r['auroc']['b'], delta=r['auroc']['delta'], lo=r['auroc']['lo'], hi=r['auroc']['hi']))
    display(C.round(4)); print('Critical-miss ranking AUROC, U vs U+E:', RESULTS['ensemble_signal']['auroc_critical_miss'])

try:
    ensemble_signal_test()
except Exception as exc:
    print('Ensemble-signal test skipped:', type(exc).__name__, exc)


## 6e. MIMIC subgroup audit (optional; needs MIMIC-IV demographics)
Sex, age band and recorded race: observed risk relative to target, referral fraction and macro-AUROC for each subgroup.

In [ ]:
# ---------------------------------------------------------------- subgroup audit on the MIMIC cohort (needs MIMIC-IV; skipped otherwise)
# Demographics are NOT in MIMIC-CXR-JPG. With credentialed MIMIC-IV access, put hosp/patients.csv.gz (and optionally hosp/admissions.csv.gz) on Drive,
# e.g. <run folder>/mimic_iv/patients.csv.gz, or set MIMIC_IV_DIR to the folder holding them. We then ask whether the observed risk targets and the
# escalation burden are equitable across sex, age band and (if admissions are present) recorded race/ethnicity. Age is MIMIC-IV's anchor_age
# (age in the anchor year), an approximation of age at the study.
MIMIC_IV_DIR = globals().get('MIMIC_IV_DIR')      # set by the upload cell or by you
def mimic_subgroups():
    cfg_run = json.loads((RUN_DIR/'config.json').read_text())
    def mapped(value):
        for prefix in ['/content/drive_fresh/MyDrive/', '/content/drive/MyDrive/']:
            if str(value).startswith(prefix):
                return RUN_DIR.parents[2]/str(value)[len(prefix):]
        return Path(value)
    jroot = mapped(cfg_run['jpg_root'])
    dirs = [Path(MIMIC_IV_DIR)] if MIMIC_IV_DIR else []
    dirs += [RUN_DIR/'mimic_iv', RUN_DIR.parent/'mimic_iv', jroot.parent/'mimic-iv'/'hosp', jroot.parent/'mimic_iv', jroot/'mimic_iv']
    found = [d for d in dirs if (d/'patients.csv.gz').is_file() or (d/'patients.csv').is_file()]
    if not found:
        # last resort: look for the MIMIC-IV tables anywhere under the project folders on Drive (shallow, 4 levels)
        for root in [RUN_DIR, RUN_DIR.parent, RUN_DIR.parents[1]]:
            if not root.is_dir():
                continue
            base_depth = len(root.parts)
            for cur, subdirs, fnames in os.walk(root):
                if len(Path(cur).parts)-base_depth >= 4:
                    subdirs[:] = []
                if any(f in ('patients.csv.gz', 'patients.csv') for f in fnames):
                    found.append(Path(cur)); break
            if found:
                break
    if not found:
        listing = sorted(p.name for p in (RUN_DIR/'mimic_iv').iterdir()) if (RUN_DIR/'mimic_iv').is_dir() else 'folder does not exist'
        raise FileNotFoundError('MIMIC-IV patients table not found. Looked in: '+'; '.join(str(x) for x in dirs)+' and below the project folders. '
                                'Contents of '+str(RUN_DIR/'mimic_iv')+': '+str(listing)+'. Put patients.csv.gz there (exact name) or set MIMIC_IV_DIR.')
    print('Using MIMIC-IV tables from', found[0])
    d = found[0]; pf = d/'patients.csv.gz' if (d/'patients.csv.gz').is_file() else d/'patients.csv'
    pts = pd.read_csv(pf, dtype={'subject_id': str})[['subject_id', 'gender', 'anchor_age']]
    cohort = pd.read_csv(RUN_DIR/'cohort_PRIVATE.csv', dtype=str)
    cohort['case'] = [hashlib.sha256(f"{cfg_run['seed']}:{p}:{s}".encode()).hexdigest()[:20] for p, s in zip(cohort.subject_id, cohort.study_id)]
    meta = cohort[['case', 'subject_id']].drop_duplicates('case').merge(pts, on='subject_id', how='left').set_index('case')
    af = d/'admissions.csv.gz' if (d/'admissions.csv.gz').is_file() else (d/'admissions.csv' if (d/'admissions.csv').is_file() else None)
    if af is not None:
        adm = pd.read_csv(af, dtype={'subject_id': str})[['subject_id', 'race']].dropna()
        adm['race_group'] = adm.race.str.upper().map(lambda r: 'WHITE' if r.startswith('WHITE') else 'BLACK' if 'BLACK' in r else 'ASIAN' if 'ASIAN' in r else 'HISPANIC' if 'HISPANIC' in r else 'OTHER/UNKNOWN')
        race = adm.groupby('subject_id').race_group.agg(lambda s: s.value_counts().index[0])
        meta['race_group'] = meta.subject_id.map(race)
    ev = EV; m_ = meta.reindex(ev['case'])
    age = m_.anchor_age.to_numpy(float)
    groups = {'sex=F': (m_.gender == 'F').to_numpy(), 'sex=M': (m_.gender == 'M').to_numpy(), 'age<40': age < 40, 'age 40-64': (age >= 40) & (age < 65), 'age>=65': age >= 65}
    if 'race_group' in m_:
        for g in ['WHITE', 'BLACK', 'ASIAN', 'HISPANIC', 'OTHER/UNKNOWN']:
            groups['race='+g] = (m_.race_group == g).to_numpy()
    rows = []
    for kind, h0 in [('fn', 'fn0'), ('severe', 'sev0')]:
        for alpha in PLAN['harm_kinds'][kind]:
            for gate in ['U', 'UDMS']:
                esc = runs[(kind, alpha, gate, 'ltt')]['esc']
                for name, mg in groups.items():
                    if mg.sum() < 100:
                        continue
                    loss = ev[h0][mg]*(~esc[mg]); ci = boot_ci(loss, PLAN['boot'], 76)
                    rows.append(dict(harm=kind, alpha=alpha, gate=gate, group=name, n=int(mg.sum()), answer_all_risk=float(ev[h0][mg].mean()), risk=ci['mean'], lo=ci['lo'], hi=ci['hi'],
                                     referral=float(esc[mg].mean()), point_target_met=bool(ci['mean'] <= alpha), upper_bound_met=bool(ci['hi'] <= alpha)))
    T = pd.DataFrame(rows)
    aur = []
    for name, mg in groups.items():
        if mg.sum() < 100:
            continue
        a_ = []
        for jj in range(len(LABELS)):
            k_ = mg & (ev['mask'][:, jj] > .5); yy = ev['y'][k_, jj]
            if min(yy.sum(), len(yy)-yy.sum()) >= 20:
                a_.append(roc_auc_score(yy, ev['p0'][k_, jj]))
        aur.append(dict(group=name, n=int(mg.sum()), macro_auroc=float(np.mean(a_)) if a_ else np.nan, labels_assessed=len(a_)))
    return T, pd.DataFrame(aur)

try:
    SGT, SGA = mimic_subgroups(); TABLES['mimic_subgroup_risk'] = SGT; TABLES['mimic_subgroup_auroc'] = SGA
    display(SGA.round(3)); display(SGT[(SGT.gate == 'UDMS')].round(4))
    full = SGT[SGT.gate == 'UDMS']
    RESULTS['mimic_subgroups'] = dict(groups=sorted(full.group.unique()), configs=int(len(full)), point_target_met=int(full.point_target_met.sum()),
                                      upper_bound_met=int(full.upper_bound_met.sum()), auroc_range=[float(SGA.macro_auroc.min()), float(SGA.macro_auroc.max())])
    print('MIMIC subgroup audit:', RESULTS['mimic_subgroups'])
except Exception as exc:
    print('MIMIC subgroup audit skipped:', type(exc).__name__, exc)


## 7. Simulated reviewer queue (RQ3)
Poisson arrivals, two reviewers and exponential service times; the override probability rises with the number of waiting cases. The workload-aware rule raises its threshold as the queue grows and is compared with a static threshold referring the same volume. All parameters are assumptions.

In [ ]:
import heapq
# ---------------------------------------------------------------- Signal 5 / RQ3: workload-aware thresholds under a simulated reviewer queue
# EVERY number in this block rests on assumptions (alert-fatigue model, service times). It is a simulation, not evidence
# about real clinicians; MIMIC-CXR-JPG has no staffing or override data.
SIM_ASSUME = dict(servers=2, service_rate=1.0, base_override=0.02, loads={'low': .6, 'medium': .9, 'overloaded': 1.2},
                  base_share=0.30, reps=10, kappas=[0.5, 1.0], override_slopes=[0.02, 0.04, 0.08])

def simulate_queue(c0hat, fn0, e, arrival_rate, theta, kappa, o1, rng, servers=2, mu=1.0, o0=0.02):
    n = len(c0hat); order = rng.permutation(n)
    arrivals = np.cumsum(rng.exponential(1/arrival_rate, n))
    free = np.zeros(servers); heap = []
    harm = np.empty(n); esc = overrides = 0; waits = []
    for t, i in zip(arrivals, order):
        while heap and heap[0] <= t:
            heapq.heappop(heap)
        waiting = max(len(heap)-servers, 0)
        if c0hat[i] >= theta*(1+kappa*waiting):
            srv = int(np.argmin(free)); start = max(t, free[srv]); service = rng.exponential(1/mu)
            free[srv] = start+service; heapq.heappush(heap, start+service); waits.append(start-t)
            esc += 1
            over = rng.random() < min(.9, o0+o1*waiting)       # alert fatigue: long queue -> more overrides
            overrides += over
            harm[i] = fn0[i] if over else e*fn0[i]
        else:
            harm[i] = fn0[i]
    return dict(harm=float(harm.mean()), share=esc/n, override_rate=overrides/max(esc, 1),
                mean_wait=float(np.mean(waits)) if waits else 0., p95_wait=float(np.quantile(waits, .95)) if waits else 0.)

c0q, fn0q = EV['c0hat_UDMS'], EV['fn0']
theta0 = float(np.quantile(c0q, 1-SIM_ASSUME['base_share']))
rows = []
for load, rho in SIM_ASSUME['loads'].items():
    rate = rho*SIM_ASSUME['servers']*SIM_ASSUME['service_rate']/SIM_ASSUME['base_share']
    for o1 in SIM_ASSUME['override_slopes']:
        for rep in range(SIM_ASSUME['reps']):
            seed = 1000*rep+7
            st = simulate_queue(c0q, fn0q, EXPERT_E, rate, theta0, 0., o1, np.random.default_rng(seed))
            rows.append(dict(load=load, o1=o1, rep=rep, policy='static', kappa=0., **st))
            for kappa in SIM_ASSUME['kappas']:
                aw = simulate_queue(c0q, fn0q, EXPERT_E, rate, theta0, kappa, o1, np.random.default_rng(seed))
                rows.append(dict(load=load, o1=o1, rep=rep, policy='workload_aware', kappa=kappa, **aw))
                # fair comparator: a static threshold that escalates the SAME share of patients
                th_m = float(np.quantile(c0q, 1-min(max(aw['share'], 1e-4), .999)))
                sm = simulate_queue(c0q, fn0q, EXPERT_E, rate, th_m, 0., o1, np.random.default_rng(seed))
                rows.append(dict(load=load, o1=o1, rep=rep, policy='static_matched_volume', kappa=kappa, **sm))
QS = pd.DataFrame(rows); TABLES['queue_simulation_raw'] = QS
agg = QS.groupby(['load', 'o1', 'policy', 'kappa'], as_index=False)[['harm', 'share', 'override_rate', 'mean_wait', 'p95_wait']].mean()
TABLES['queue_simulation'] = agg
display(agg[agg.o1 == 0.04].round(4))
# paired verdicts: workload-aware vs static at the same escalation volume
rows = []
for load in SIM_ASSUME['loads']:
    for o1 in SIM_ASSUME['override_slopes']:
        for kappa in SIM_ASSUME['kappas']:
            a = QS[(QS['load'] == load) & (QS.o1 == o1) & (QS.policy == 'workload_aware') & (QS.kappa == kappa)].sort_values('rep')
            b = QS[(QS['load'] == load) & (QS.o1 == o1) & (QS.policy == 'static_matched_volume') & (QS.kappa == kappa)].sort_values('rep')
            s = QS[(QS['load'] == load) & (QS.o1 == o1) & (QS.policy == 'static')].sort_values('rep')
            d_h = a.harm.values-b.harm.values
            rows.append(dict(load=load, override_slope=o1, kappa=kappa, harm_aware=a.harm.mean(), harm_static_same_volume=b.harm.mean(),
                             delta_harm=d_h.mean(), reps_aware_better=int((d_h < 0).sum()), reps=len(d_h),
                             override_aware=a.override_rate.mean(), override_static=s.override_rate.mean(),
                             wait_p95_aware=a.p95_wait.mean(), wait_p95_static=s.p95_wait.mean(),
                             volume_aware=a.share.mean(), volume_static=s.share.mean()))
QV = pd.DataFrame(rows); TABLES['queue_verdicts'] = QV
display(QV.round(4))
RESULTS['rq3_simulation'] = dict(assumptions=SIM_ASSUME,
    settings=int(len(QV)), aware_lower_harm_same_volume=int((QV.delta_harm < 0).sum()),
    aware_lower_override=int((QV.override_aware < QV.override_static).sum()),
    note='Simulation under assumed alert-fatigue model; not evidence about real clinicians.')
print(RESULTS['rq3_simulation'])


## 7b. Second labeler (NegBio)
Agreement with the CheXpert labels and observed risk when predictions are rescored against NegBio. Skipped if the NegBio file is absent.

In [ ]:
# ---------------------------------------------------------------- robustness to the label source: CheXpert vs NegBio (second rule-based labeler)
# All outcomes above use the CheXpert labeler. MIMIC-CXR-JPG also ships labels from NegBio. The SAME trained models and gate scores are re-scored
# against NegBio labels and we ask: (a) how much do the two labelers agree on these studies; (b) does the guarantee survive when the threshold is
# recalibrated on NegBio labels; (c) does it survive when the gate keeps its CheXpert calibration (labeler shift)?
# Put mimic-cxr-2.0.0-negbio.csv.gz next to mimic-cxr-2.0.0-chexpert.csv.gz on Drive (or set NEGBIO_PATH). Skips itself if it is missing.
NEGBIO_PATH = None                          # optional explicit path, e.g. Path('/content/drive_fresh/MyDrive/.../mimic-cxr-2.0.0-negbio.csv.gz')
def label_robustness():
    cfg_run = json.loads((RUN_DIR/'config.json').read_text())
    def mapped(value):
        for prefix in ['/content/drive_fresh/MyDrive/', '/content/drive/MyDrive/']:
            if str(value).startswith(prefix):
                return RUN_DIR.parents[2]/str(value)[len(prefix):]
        return Path(value)
    path = Path(NEGBIO_PATH) if NEGBIO_PATH else mapped(cfg_run['jpg_root'])/'mimic-cxr-2.0.0-negbio.csv.gz'
    if not path.is_file():
        raise FileNotFoundError('NegBio label file not found: '+str(path))
    cohort = pd.read_csv(RUN_DIR/'cohort_PRIVATE.csv', dtype=str)
    cohort['case'] = [hashlib.sha256(f"{cfg_run['seed']}:{p}:{s}".encode()).hexdigest()[:20] for p, s in zip(cohort.subject_id, cohort.study_id)]
    lab = pd.read_csv(path, dtype={'subject_id': str, 'study_id': str})
    j = cohort[['case', 'subject_id', 'study_id']].merge(lab[['subject_id', 'study_id']+LABELS], on=['subject_id', 'study_id'], how='left', indicator=True)
    j = j[j['_merge'] == 'both'].drop_duplicates('case').set_index('case')
    raw = j[LABELS].to_numpy(float)
    alt_y = (np.nan_to_num(raw, nan=0) > 0).astype(float); alt_m = (raw != -1).astype(float)
    pos = {c: i for i, c in enumerate(j.index)}
    # (a) agreement between the two labelers on the evaluation patients
    idx = np.array([pos.get(c, -1) for c in EVB['case']]); okc = idx >= 0
    rows = []
    for jj, label in enumerate(LABELS):
        both = okc & (EVB['mask'][:, jj] > .5) & (np.where(okc, alt_m[np.where(okc, idx, 0), jj], 0) > .5)
        a = EVB['y'][both, jj] > .5; b = alt_y[idx[both], jj] > .5
        po = float((a == b).mean()); pe = float(a.mean()*b.mean()+(1-a.mean())*(1-b.mean()))
        rows.append(dict(label=label, n=int(both.sum()), prevalence_chexpert=float(a.mean()), prevalence_negbio=float(b.mean()),
                         observed_agreement=po, cohen_kappa=float((po-pe)/(1-pe)) if pe < 1 else float('nan')))
    AGR = pd.DataFrame(rows); TABLES['label_agreement_chexpert_vs_negbio'] = AGR; display(AGR.round(3))
    RESULTS['label_agreement'] = dict(patients_matched=int(okc.sum()), patients=int(len(okc)), macro_kappa=float(AGR.cohen_kappa.mean()))
    # (b)+(c) guarantee under a different labeler, burden and critical-miss targets
    rows = []
    for kind, alphas in PLAN['harm_kinds'].items():
        pre = 'sc_fn_' if kind == 'fn' else 'sc_sv_'; hk = 'fn' if kind == 'fn' else 'severe'; h_old = 'fn0' if kind == 'fn' else 'sev0'
        for alpha in alphas:
            for gate in ['U', 'UDMS']:
                rec = {'recal_new': [], 'keep_old': [], 'ref_new': [], 'ref_old': []}
                for k in range(K):
                    F = FOLDS[k]; cal = get_cal(k, PRIMARY)
                    def alt_outcome(role):
                        ii = np.array([pos.get(c, -1) for c in F[f'{role}_case']]); ok = ii >= 0
                        return ok, outcome(F[f'{role}_{PRIMARY}_p0'][ok], alt_y[ii[ok]], alt_m[ii[ok]])
                    ok_c, o_c = alt_outcome('cal'); ok_e, o_e = alt_outcome('ev')
                    s_cal = cal[pre+gate]; s_ev = F[f'ev_{PRIMARY}_'+pre+gate]
                    t_old, _ = calibrate_gate(s_cal, cal[h_old], alpha, PLAN['delta'], 'ltt')
                    t_new, _ = calibrate_gate(s_cal[ok_c], o_c[hk], alpha, PLAN['delta'], 'ltt')
                    se = s_ev[ok_e]
                    rec['keep_old'].append(float((o_e[hk]*(se < t_old)).mean())); rec['recal_new'].append(float((o_e[hk]*(se < t_new)).mean()))
                    rec['ref_old'].append(float((se >= t_old).mean())); rec['ref_new'].append(float((se >= t_new).mean()))
                rows.append(dict(harm=kind, alpha=alpha, gate=gate, risk_kept_chexpert_calibration=np.mean(rec['keep_old']), referral_kept=np.mean(rec['ref_old']),
                                 risk_recalibrated_on_negbio=np.mean(rec['recal_new']), referral_recalibrated=np.mean(rec['ref_new']),
                                 target_met_if_recalibrated=bool(np.mean(rec['recal_new']) <= alpha),
                                 target_met_if_labeler_shifts=bool(np.mean(rec['keep_old']) <= alpha)))
    return pd.DataFrame(rows)

try:
    LBL = label_robustness(); TABLES['label_source_robustness'] = LBL; display(LBL.round(4))
    RESULTS['label_robustness'] = dict(recalibrated_met=int(LBL.target_met_if_recalibrated.sum()),
                                       shifted_met=int(LBL.target_met_if_labeler_shifts.sum()), configs=int(len(LBL)),
                                       shifted_overshoot_max=float((LBL.risk_kept_chexpert_calibration-LBL.alpha).max()))
    print('Label-source robustness:', RESULTS['label_robustness'], '| agreement:', RESULTS['label_agreement'])
except Exception as exc:
    print('Label-source robustness skipped:', type(exc).__name__, exc)


## 8. Additional MIMIC cohort (2,000 patients; cached features)
Reads cached features for 2,000 patients outside the development cohort. Each feature file is matched to its patient by re-running the saved classifier and requiring an exact match to its stored predictions; the cell stops otherwise. Needs the CheXpert label file on Drive. Skipped if the cached cohort is absent.

In [ ]:
# ---------------------------------------------------------------- Untouched confirmation on the cached 2,000 fresh patients (OPTIONAL)
# V13 already extracted frozen features for 2,000 patients who are NOT among the 5,994 above (one frontal study + context each) and cached
# them as small .npz files. We re-read those cached features only: no download, no image/report/encoder work. The identity of each cached
# feature file is recovered by re-running the frozen V13 classifier on it and matching the saved V13 predictions exactly.
# A lateral view was not extracted for these patients, so ASK-LATERAL cannot be tested here; ASK-CONTEXT and the guarantee can.
class LabelHeads:                          # stubs so the saved V13 classifier pickle can be read
    def predict(self, x):
        return safe_prob(np.column_stack([np.full(len(x), m) if isinstance(m, float) else m.predict_proba(x)[:, 1] for m in self.models]))
class Stacker:
    pass

def load_fresh():
    V13 = RUN_DIR/V13_REL
    if not (V13/'locked_protocol.json').is_file():
        raise FileNotFoundError('V13 folder not found: '+str(V13))
    pred = read_npz(V13/'new_patient_predictions_PRIVATE.npz')
    cfg_run = json.loads((RUN_DIR/'config.json').read_text())
    def mapped(value):
        for prefix in ['/content/drive_fresh/MyDrive/', '/content/drive/MyDrive/']:
            if str(value).startswith(prefix):
                return RUN_DIR.parents[2]/str(value)[len(prefix):]
        return Path(value)
    label_path = mapped(cfg_run['jpg_root'])/'mimic-cxr-2.0.0-chexpert.csv.gz'
    if not label_path.is_file():
        raise FileNotFoundError('Reference label file not found: '+str(label_path))
    files = sorted(f for sub in (V13/'features').iterdir() if sub.is_dir() for f in sub.glob('*.npz'))
    feats = [read_npz(f) for f in files]
    img = np.stack([z['bio'] for z in feats]).astype(np.float32)
    txt = np.stack([z['text'] for z in feats]).astype(np.float32)
    present = np.array([bool(z['present']) for z in feats])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        with (RUN_DIR/'frozen_models_PRIVATE.pkl').open('rb') as h:
            bundle = pickle.load(h)
    pi = bundle['image_head'].predict(img); pt = bundle['text_head'].predict(txt)
    raw = np.where(present[:, None], (1-bundle['text_weight'])*pi+bundle['text_weight']*pt, pi)
    frozen = apply_temperature(raw, np.asarray(bundle['temperatures']))
    target = pred['biomedclip']
    match = np.full(len(frozen), -1)
    for s in range(0, len(frozen), 250):
        d = np.abs(frozen[s:s+250, None, :]-target[None, :, :]).max(2)
        j = d.argmin(1); ok = d[np.arange(len(j)), j] < 1e-4
        block = match[s:s+250]; block[ok] = j[ok]
    keep = match >= 0
    if keep.sum() < 500 or len(set(match[keep])) != keep.sum():
        raise RuntimeError(f'Only {int(keep.sum())} cached feature files could be matched uniquely to the frozen V13 predictions.')
    sub = pred['subject_id'].astype(str)[match[keep]]; stu = pred['study_id'].astype(str)[match[keep]]
    labs = pd.read_csv(label_path, dtype={'subject_id': str, 'study_id': str})
    j = pd.DataFrame({'subject_id': sub, 'study_id': stu}).merge(labs[['subject_id', 'study_id']+LABELS], on=['subject_id', 'study_id'], how='left', indicator=True)
    matched = j['_merge'].eq('both').to_numpy()
    raw_l = j[LABELS].to_numpy(float)
    mask_f = (raw_l != -1).astype(float)          # historical CheXpert convention: -1 uncertain is masked, blank = negative
    y_f = (np.nan_to_num(raw_l, nan=0) > 0).astype(float)
    ok = matched & (mask_f.sum(1) > 0)
    return dict(image=img[keep][ok], text=txt[keep][ok], present=present[keep][ok], y=y_f[ok], mask=mask_f[ok], subject=sub[ok])

FRESH = None
try:
    FRESH = load_fresh()
    print(f"Fresh cohort recovered from cached features: {len(FRESH['subject']):,} patients with labels (exact match to frozen V13 predictions).")
except Exception as exc:
    print('Fresh-cohort confirmation skipped:', type(exc).__name__, exc)

def fresh_confirmation():
    nF = len(FRESH['subject']); tz = (FRESH['text']*FRESH['present'][:, None]).astype(np.float32)
    u_f = hash_uniform(FRESH['subject'], 'ctxfresh'); m_ = int(re.sub(r'\D', '', PRIMARY))/100
    avail = FRESH['present'] & (u_f >= m_); ctx_ok = FRESH['present'] & ~avail; lat_ok = np.zeros(nF, bool)
    per_fold = []
    for k in range(K):
        with (OUT/f'fold{k}_bundle_PRIVATE.pkl').open('rb') as h:
            B = pickle.load(h)
        Hh, T_ = B['H'], B['TEMP']
        img = apply_temperature(Hh['img'].predict(FRESH['image']), T_['img'])
        ic = apply_temperature(Hh['ic'].predict(np.hstack([FRESH['image'], tz])), T_['ic'])
        tx = apply_temperature(Hh['txt'].predict(tz), T_['txt'])
        p0 = np.where(avail[:, None], ic, img); pc = np.where(FRESH['present'][:, None], ic, img)
        bl = signal_blocks(p0, img, tx, avail, FRESH['image'], tz, B['aux'], B['q'])
        o0, oc = outcome(p0, FRESH['y'], FRESH['mask']), outcome(pc, FRESH['y'], FRESH['mask'])
        sc = dict(entropy=bl['U'][:, 14])
        for g, groups in GATE_SETS.items():
            X = design(bl, groups)
            sc['fn_'+g] = B['reg']['fn_'+g].predict(X); sc['sv_'+g] = B['reg']['sv_'+g].predict(X)
            sc['c0_'+g] = np.maximum(B['reg']['c0_'+g].predict(X), 0.)
        sc['gc'] = B['reg']['gc'].predict(design(bl, GATE_SETS['UDMS']))
        per_fold.append((o0, oc, sc))
    RESULTS['fresh_n'] = nF
    RESULTS['fresh_answer_all'] = dict(fn=float(per_fold[0][0]['fn'].mean()), severe=float(per_fold[0][0]['severe'].mean()))
    rows = []; esc_store = {}
    for kind, alphas in PLAN['harm_kinds'].items():
        h0 = HARM[kind][0]; key = 'fn' if kind == 'fn' else 'severe'
        for alpha in alphas:
            for gate in ['entropy', 'U', 'UD', 'UDM', 'UDMS']:
                loss_f, esc_m = [], []
                for k in range(K):
                    o0, oc, sc = per_fold[k]; cal = get_cal(k, PRIMARY)
                    s_cal = cal['sc_entropy'] if gate == 'entropy' else cal[HARM[kind][3]+gate]
                    s_new = sc['entropy'] if gate == 'entropy' else sc[('fn_' if kind == 'fn' else 'sv_')+gate]
                    t, _ = calibrate_gate(s_cal, cal[h0], alpha, PLAN['delta'], 'ltt')
                    e_k = s_new >= t; esc_m.append(e_k); loss_f.append(o0[key]*(~e_k))
                loss_i = np.mean(loss_f, axis=0); esc_i = np.mean(esc_m, axis=0); ci = boot_ci(loss_i, PLAN['boot'], 51)
                esc_store[(kind, alpha, gate)] = esc_i
                rows.append(dict(harm=kind, alpha=alpha, gate=gate, referral=float(esc_i.mean()), risk=ci['mean'], risk_lo=ci['lo'],
                                 risk_hi=ci['hi'], folds_within_target=int(np.sum([l.mean() <= alpha for l in loss_f])),
                                 target_met=bool(ci['mean'] <= alpha)))
    FRT = pd.DataFrame(rows); TABLES['fresh_cohort_gate'] = FRT; display(FRT.round(4))
    cons = []
    for kind, alphas in PLAN['harm_kinds'].items():
        for alpha in alphas:
            ci = boot_ci(esc_store[(kind, alpha, 'UDMS')]-esc_store[(kind, alpha, 'U')], PLAN['boot'], 52)
            cons.append(dict(harm=kind, alpha=alpha, delta_referral_full_vs_U=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci), p=ci['p']))
    FCON = pd.DataFrame(cons); FCON['p_holm'] = holm(FCON.p.to_numpy()); TABLES['fresh_cohort_contrast'] = FCON; display(FCON.round(4))
    # ASK-CONTEXT on the fresh cohort at the primary review budget
    p_ = dict(P_ASK, allow_ctx=True, allow_lat=False); p2 = dict(P_ASK, allow_ctx=False, allow_lat=False)
    d_cost = []; asks = []
    for k in range(K):
        o0, oc, sc = per_fold[k]; zero = np.zeros(nF)
        c0 = o0['fn']+FPW*o0['fp']; cc = oc['fn']+FPW*oc['fp']
        a3 = pick_lambda4(sc['c0_UDMS'], sc['gc'], zero, ctx_ok, lat_ok, EXPERT_E, p_, PLAN['primary_budget'], LAM)[1]
        a2 = pick_lambda4(sc['c0_UDMS'], sc['gc'], zero, ctx_ok, lat_ok, EXPERT_E, p2, PLAN['primary_budget'], LAM)[1]
        d_cost.append(realised4(a3, c0, cc, zero, EXPERT_E, p_)-realised4(a2, c0, cc, zero, EXPERT_E, p2)); asks.append((a3 == 2).mean())
    ci = boot_ci(np.mean(d_cost, axis=0), PLAN['boot'], 53)
    RESULTS['fresh_ask_context'] = dict(delta_cost=ci['mean'], lo=ci['lo'], hi=ci['hi'], verdict=verdict(ci), ask_rate=float(np.mean(asks)))
    v = FRT[FRT.gate == 'UDMS']
    RESULTS['fresh_confirmation'] = dict(patients=nF, configs=int(len(v)), target_met=int(v.target_met.sum()),
                                         referral_full_vs_U=FCON.to_dict('records'))
    print('Fresh-cohort guarantee (full gate):', RESULTS['fresh_confirmation']['target_met'], 'of', len(v), 'targets met.')
    print('Fresh-cohort ASK-CONTEXT at the primary budget:', RESULTS['fresh_ask_context'])

if FRESH is not None:
    try:
        fresh_confirmation()
    except Exception as exc:
        print('Fresh-cohort confirmation stopped:', type(exc).__name__, exc)


## 8b. External comparison on NIH ChestX-ray14 (optional; the only download step)
Encodes one frontal image per patient with the same frozen BiomedCLIP encoder, applies the MIMIC-fitted heads, referral models and conformal quantiles unchanged, and compares the MIMIC-calibrated threshold with a threshold recalibrated on a local NIH calibration split. The cohort uses the first four NIH image archives and is capped at 6,000 patients.

In [ ]:
# ---------------------------------------------------------------- EXTERNAL VALIDATION, step 1 of 2 (OPTIONAL, CPU, ~25-60 min on free Colab)
# NIH ChestX-ray14 (public, no credentials). We encode ONE frontal image per patient with the SAME frozen BiomedCLIP image encoder used for the
# MIMIC snapshot, and cache the embeddings to Drive so this step runs once. It downloads one 2 GB archive and the BiomedCLIP weights (~0.4 GB):
# this is the only part of the notebook that downloads anything, and it is skipped if the cached embeddings already exist.
N_ZIPS = int(os.environ.get('ARG_N_ZIPS', 4))             # number of 2 GB NIH image archives (each ~1,300 patients with a PA/AP image); 4 = ~5,000 patients, ~1.5-2 h CPU encoding; use 2 for a quicker run
N_EXT = 6000                                              # cap on NIH patients to encode (CPU time scales linearly)
EXT_DIR = RUN_DIR/'external_nih'; EXT_DIR.mkdir(exist_ok=True)
EXT_NPZ = EXT_DIR/'nih_embeddings.npz'
NIH_SHARED = {'No Finding': 'No Finding', 'Cardiomegaly': 'Cardiomegaly', 'Edema': 'Edema', 'Consolidation': 'Consolidation',
              'Pneumonia': 'Pneumonia', 'Atelectasis': 'Atelectasis', 'Pneumothorax': 'Pneumothorax', 'Effusion': 'Pleural Effusion'}
def nih_encode():
    import zipfile, io, subprocess as sp
    work = Path(os.environ['NIH_WORK']) if os.environ.get('NIH_WORK') else (Path('/content/nih_work') if Path('/content').exists() else EXT_DIR/'work')
    work.mkdir(parents=True, exist_ok=True)
    for pkg, mod in [('open_clip_torch', 'open_clip'), ('huggingface_hub', 'huggingface_hub'), ('transformers', 'transformers'), ('timm', 'timm')]:
        if importlib.util.find_spec(mod) is None:
            sp.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
    import torch, open_clip
    from PIL import Image
    base = 'https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main/data/'
    entry_csv = work/'Data_Entry_2017_v2020.csv'
    if not entry_csv.is_file():
        sp.check_call(['curl', '-sL', '-o', str(entry_csv), base+'Data_Entry_2017_v2020.csv'])
    zips = {}; names = {}
    for zi in range(1, N_ZIPS+1):
        zp = work/f'images_{zi:03d}.zip'
        if not zp.is_file() or zp.stat().st_size < 1_000_000_000:
            sp.check_call(['curl', '-sL', '-C', '-', '-o', str(zp), base+f'images/images_{zi:03d}.zip'])
        zips[zi] = zipfile.ZipFile(zp)
        names.update({Path(n).name: (zi, n) for n in zips[zi].namelist() if n.lower().endswith('.png')})
    entry = pd.read_csv(entry_csv)
    df = entry[entry['Image Index'].isin(names) & entry['View Position'].isin(['PA', 'AP'])].sort_values(['Patient ID', 'Follow-up #'])
    df = df.drop_duplicates('Patient ID').copy()
    df['h'] = [hashlib.sha256(f'nih:{p}'.encode()).hexdigest() for p in df['Patient ID']]
    df = df.sort_values('h').head(N_EXT).reset_index(drop=True)
    enc_pin = json.loads((RUN_DIR/'methodology_v13_final/d8228aba22e5bffd9bc8/encoder_pin.json').read_text()) if (RUN_DIR/'methodology_v13_final/d8228aba22e5bffd9bc8/encoder_pin.json').is_file() else {}
    repo = 'microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
    try:
        from huggingface_hub import snapshot_download
        local = snapshot_download(repo, revision=enc_pin.get('biomedclip_commit')) if enc_pin.get('biomedclip_commit') else None
        model, _, preprocess = open_clip.create_model_and_transforms('local-dir:'+local if local else 'hf-hub:'+repo)
    except Exception as exc:
        print('Pinned-revision load failed ('+type(exc).__name__+'); using the default revision of', repo)
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:'+repo)
    model.eval(); torch.set_num_threads(os.cpu_count() or 2)
    feats = []; t0 = time.time()
    with torch.inference_mode():
        for s in range(0, len(df), 32):
            batch = []
            for nm in df['Image Index'][s:s+32]:
                zi, member = names[nm]
                with zips[zi].open(member) as fh:
                    batch.append(preprocess(Image.open(io.BytesIO(fh.read())).convert('RGB')))
            feats.append(model.encode_image(torch.stack(batch)).float().numpy())
            if (s//32) % 10 == 0:
                print(f'  encoded {min(s+32, len(df))}/{len(df)}  ({time.time()-t0:.0f}s)', flush=True)
    feats = np.concatenate(feats).astype(np.float32)
    y = np.zeros((len(df), len(LABELS))); m = np.zeros((len(df), len(LABELS)))
    findings = [set(str(f).split('|')) for f in df['Finding Labels']]
    for nih_name, mimic_name in NIH_SHARED.items():
        j = LABELS.index(mimic_name); m[:, j] = 1; y[:, j] = [float(nih_name in f) for f in findings]
    atomic_npz(EXT_NPZ, n_zips=np.array(N_ZIPS), feats=feats, y=y, mask=m, patient=df['Patient ID'].astype(str).to_numpy(str), image=df['Image Index'].to_numpy(str),
               age=df['Patient Age'].to_numpy(float), sex=df['Patient Gender'].astype(str).to_numpy(str))
    print('Saved', EXT_NPZ, feats.shape)
def _cached_ok():
    if not EXT_NPZ.is_file():
        return False
    with np.load(EXT_NPZ) as zz:
        return 'n_zips' in zz.files and int(zz['n_zips']) == N_ZIPS
if _cached_ok():
    print('External embeddings already cached for N_ZIPS =', N_ZIPS, ':', EXT_NPZ)
else:
    try:
        nih_encode()
    except Exception as exc:
        print('External encoding skipped:', type(exc).__name__, exc)


In [ ]:
# ---------------------------------------------------------------- EXTERNAL VALIDATION, step 2 of 2: NIH ChestX-ray14 (different hospital, labeler and image format)
# The MIMIC-trained heads, gate models and conformal quantiles are applied UNCHANGED to NIH embeddings (image only: no clinical context exists
# there, so this matches the "context withheld" case). Only 8 labels exist in both datasets; the others are masked. Two questions:
#  (a) does a gate calibrated on MIMIC keep its risk guarantee after the dataset shift?   (b) does RE-calibrating only the escalation threshold on a
#  small NIH calibration split restore it, and does the full gate still need no more escalations than the baselines?
def nih_external():
    z = read_npz(EXT_NPZ); feats, y, m = z['feats'], z['y'], z['mask']
    n = len(feats); u = hash_uniform(z['patient'], 'nihsplit'); cal_i = u < 1/3; te_i = ~cal_i
    shared = np.flatnonzero(m[0] > 0)
    SEV8 = [(LABELS[j], SEVERITY[j]) for j in shared]
    per_fold = []
    for k in range(K):
        with (OUT/f'fold{k}_bundle_PRIVATE.pkl').open('rb') as h:
            B = pickle.load(h)
        img = apply_temperature(B['H']['img'].predict(feats), B['TEMP']['img'])
        tdim = B['aux']['txt_scaler'].mean_.shape[0]; zero_t = np.zeros((n, tdim), np.float32)
        bl = signal_blocks(img, img, img, np.zeros(n, bool), feats, zero_t, B['aux'], B['q'])
        o0 = outcome(img, y, m)
        sc = dict(entropy=bl['U'][:, 14], ambiguity=bl['U'][:, 15])
        for g in ['U', 'UDMS']:
            X = design(bl, GATE_SETS[g]); sc['fn_'+g] = B['reg']['fn_'+g].predict(X); sc['sv_'+g] = B['reg']['sv_'+g].predict(X)
        per_fold.append((img, o0, sc))
    img_mean = np.mean([p[0] for p in per_fold], axis=0)
    # transfer quality of the MIMIC-trained image head on the shared labels
    rows = []
    for j in shared:
        yy = y[:, j]
        rows.append(dict(label=LABELS[j], n=n, prevalence=float(yy.mean()), auroc_image_only=float(roc_auc_score(yy, img_mean[:, j]))))
    AUC = pd.DataFrame(rows); TABLES['external_nih_transfer_auroc'] = AUC; display(AUC.round(3))
    base_fn = float(per_fold[0][1]['fn'][te_i].mean()); base_sev = float(per_fold[0][1]['severe'][te_i].mean())
    ALPHA = dict(fn=[round(base_fn*f_, 4) for f_ in (0.25, 0.5, 0.75)], severe=[round(base_sev*f_, 3) for f_ in (0.4, 0.6, 0.8)])
    print(f'NIH answer-all burden (test split): fn {base_fn:.4f}, severe {base_sev:.3f}; targets {ALPHA}')
    rows = []; esc_store = {}; per_patient = {}
    for kind, alphas in ALPHA.items():
        hk = 'fn' if kind == 'fn' else 'severe'; pre = 'fn_' if kind == 'fn' else 'sv_'
        for alpha in alphas:
            for gate in ['entropy', 'ambiguity', 'U', 'UDMS']:
                loss_old, loss_new, esc_old, esc_new = [], [], [], []
                for k in range(K):
                    img, o0, sc = per_fold[k]; cal = get_cal(k, 'mcar60')
                    s_nih = sc[gate] if gate in ('entropy', 'ambiguity') else sc[pre+gate]
                    s_src = cal['sc_'+gate] if gate in ('entropy', 'ambiguity') else cal[('sc_fn_' if kind == 'fn' else 'sc_sv_')+gate]
                    h_src = cal['fn0' if kind == 'fn' else 'sev0']; keep = ~cal['avail']          # MIMIC calibration rows whose context was unavailable
                    t_old, _ = calibrate_gate(s_src[keep], h_src[keep], alpha, PLAN['delta'], 'ltt')
                    t_new, _ = calibrate_gate(s_nih[cal_i], o0[hk][cal_i], alpha, PLAN['delta'], 'ltt')
                    e_o, e_n = s_nih[te_i] >= t_old, s_nih[te_i] >= t_new
                    loss_old.append(o0[hk][te_i]*(~e_o)); loss_new.append(o0[hk][te_i]*(~e_n)); esc_old.append(e_o); esc_new.append(e_n)
                lo_, ln_ = np.mean(loss_old, axis=0), np.mean(loss_new, axis=0)
                ci_o, ci_n = boot_ci(lo_, PLAN['boot'], 71), boot_ci(ln_, PLAN['boot'], 72)
                esc_store[(kind, alpha, gate)] = np.mean(esc_new, axis=0)
                per_patient[(kind, alpha, gate)] = (ln_, lo_, np.mean(esc_new, axis=0), np.mean(esc_old, axis=0))
                rows.append(dict(harm=kind, alpha=alpha, gate=gate,
                                 referral_mimic_calibrated=float(np.mean(esc_old)), risk_mimic_calibrated=ci_o['mean'], risk_hi_mimic=ci_o['hi'], target_met_mimic=bool(ci_o['mean'] <= alpha),
                                 referral_nih_recalibrated=float(np.mean(esc_new)), risk_nih_recalibrated=ci_n['mean'], risk_hi_nih=ci_n['hi'], target_met_nih=bool(ci_n['mean'] <= alpha)))
    G = pd.DataFrame(rows); TABLES['external_nih_gate'] = G; display(G.round(4))
    cons = []
    for kind, alphas in ALPHA.items():
        for alpha in alphas:
            for other in ['U', 'entropy', 'ambiguity']:
                ci = boot_ci(esc_store[(kind, alpha, 'UDMS')]-esc_store[(kind, alpha, other)], PLAN['boot'], 73)
                cons.append(dict(harm=kind, alpha=alpha, comparison=f'Full gate vs {other}', delta_referral=ci['mean'], lo=ci['lo'], hi=ci['hi'], p=ci['p'], mde80=ci['mde80']))
    C = pd.DataFrame(cons); C['p_holm'] = holm(C.p.to_numpy()); C['verdict_holm'] = np.where(C.p_holm < .05, np.where(C.delta_referral < 0, 'better', 'worse'), 'inconclusive')
    TABLES['external_nih_contrast'] = C; display(C.round(4))
    # ---- subgroup audit (sex, age) on the NIH test split: does the risk target hold in every subgroup?
    age, sex = z['age'], z['sex']
    groups = {'sex=F': sex == 'F', 'sex=M': sex == 'M', 'age<40': age < 40, 'age 40-59': (age >= 40) & (age < 60), 'age>=60': age >= 60}
    rows = []
    for (kind, alpha, gate), (ln_i, lo_i, en_i, eo_i) in per_patient.items():
        if gate not in ('U', 'UDMS'):
            continue
        for name, mg_all in groups.items():
            mg = mg_all[te_i]
            if mg.sum() < 30:
                continue
            ci = boot_ci(ln_i[mg], PLAN['boot'], 74)
            rows.append(dict(harm=kind, alpha=alpha, gate=gate, group=name, n=int(mg.sum()), risk_recalibrated=ci['mean'], lo=ci['lo'], hi=ci['hi'],
                             referral_recalibrated=float(en_i[mg].mean()), risk_mimic_calibrated=float(lo_i[mg].mean()), referral_mimic_calibrated=float(eo_i[mg].mean()),
                             point_target_met=bool(ci['mean'] <= alpha), upper_bound_met=bool(ci['hi'] <= alpha)))
    SG = pd.DataFrame(rows); TABLES['external_nih_subgroups'] = SG; display(SG[SG.gate == 'UDMS'].round(4))
    gaps = []
    for (kind, alpha, gate), (ln_i, lo_i, en_i, eo_i) in per_patient.items():
        if gate != 'UDMS':
            continue
        for g1, g2 in [('sex=F', 'sex=M'), ('age<40', 'age>=60')]:
            m1, m2 = groups[g1][te_i], groups[g2][te_i]
            if m1.sum() < 30 or m2.sum() < 30:
                continue
            rng_ = np.random.default_rng(75); d = []
            for _ in range(min(PLAN['boot'], 2000)):
                d.append(ln_i[m1][rng_.integers(0, m1.sum(), m1.sum())].mean()-ln_i[m2][rng_.integers(0, m2.sum(), m2.sum())].mean())
            lo_, hi_ = np.quantile(d, [.025, .975])
            gaps.append(dict(harm=kind, alpha=alpha, comparison=f'{g1} minus {g2}', risk_difference=float(ln_i[m1].mean()-ln_i[m2].mean()), lo=float(lo_), hi=float(hi_),
                             referral_difference=float(en_i[m1].mean()-en_i[m2].mean())))
    GP = pd.DataFrame(gaps); TABLES['external_nih_subgroup_gaps'] = GP; display(GP.round(4))
    RESULTS['external_nih_subgroups'] = dict(groups=list(groups), rows=int(len(SG)), point_target_met=int(SG.point_target_met.sum()), upper_bound_met=int(SG.upper_bound_met.sum()),
                                             gaps_with_interval_excluding_zero=int(((GP.lo > 0) | (GP.hi < 0)).sum()), gaps=int(len(GP)))
    fu = G[G.gate == 'UDMS']
    RESULTS['external_nih'] = dict(patients=int(n), shared_labels=[LABELS[j] for j in shared], macro_auroc=float(AUC.auroc_image_only.mean()),
                                   configs=int(len(fu)), target_met_mimic_calibrated=int(fu.target_met_mimic.sum()), target_met_nih_recalibrated=int(fu.target_met_nih.sum()),
                                   contrasts=C.to_dict('records'))
    print('External NIH:', {k_: v_ for k_, v_ in RESULTS['external_nih'].items() if k_ != 'contrasts'})

if EXT_NPZ.is_file():
    try:
        nih_external()
    except Exception as exc:
        print('External NIH evaluation stopped:', type(exc).__name__, exc)
else:
    print('External NIH evaluation skipped: no cached embeddings (run the encoding cell).')


## 9. Figures, summary and exports
Writes every table as CSV and to a summary workbook.

In [ ]:
# ---------------------------------------------------------------- figures (colour-blind-safe palette, thin marks)
PAL = {'U': '#0072B2', 'UD': '#E69F00', 'UDM': '#009E73', 'UDMS': '#CC79A7', 'entropy': '#56B4E9', 'oracle': '#7a7a7a', 'ctx': '#D55E00'}
fig_dir = OUT/'figures'; fig_dir.mkdir(exist_ok=True)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25, 'font.size': 9, 'axes.titlesize': 10})
fig, axes = plt.subplots(2, 3, figsize=(17, 9.5))
for ax, kind, ttl in [(axes[0, 0], 'fn', 'Workload needed at a guaranteed severity-weighted miss burden'),
                      (axes[0, 1], 'severe', 'Workload needed at a guaranteed critical-miss rate')]:
    for g in ['entropy', 'U', 'UD', 'UDM', 'UDMS', 'oracle']:
        r = GT[(GT.harm == kind) & (GT.gate == g) & (GT.method == 'ltt')].sort_values('alpha')
        ax.plot(r.alpha, r.referral, 'o--' if g == 'oracle' else 'o-', lw=1.5, ms=4, color=PAL[g], label=g)
    ax.set(xlabel='target risk alpha (finite-sample guarantee)', ylabel='fraction escalated to clinician', title=ttl); ax.legend(fontsize=7, frameon=False)
ax = axes[0, 2]
for p, c, mk in [('binary_U', PAL['U'], 'o'), ('binary_UDMS', PAL['UDMS'], 's'), ('ask_ctx_UDMS', PAL['ctx'], '^'), ('ask_both_UDMS', '#000000', 'D'),
                 ('binary_entropy', PAL['entropy'], 'o'), ('random', '#999999', 'x')]:
    r = FR[FR.policy == p].sort_values('budget'); ax.plot(r.workload, r.cost, marker=mk, lw=1.5, ms=4, color=c, label=p)
ax.set(xlabel='mean workload per patient', ylabel='realised cost (FN + FP burden)', title='Safety vs workload frontier (primary scenario)'); ax.legend(fontsize=7, frameon=False)
ax = axes[1, 0]
c = CLS.sort_values('gain_context'); xs = np.arange(len(c))
ax.barh(xs-0.2, c.gain_context, height=.38, color=PAL['ctx'], label='+ clinical context'); ax.barh(xs+0.2, c.gain_lateral, height=.38, color=PAL['U'], label='+ lateral view')
ax.set_yticks(xs, c.label, fontsize=7); ax.axvline(0, color='k', lw=.8); ax.set(xlabel='AUROC gain over image only', title='What each information request is worth'); ax.legend(frameon=False, fontsize=7)
ax = axes[1, 1]
e_ = ECON[(ECON.request == 'context') & (ECON.budget == 0.3)].pivot_table(index='a', columns='f', values='delta_cost')
lim = np.abs(e_.values).max()+1e-12
im = ax.imshow(e_.values, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
ax.set_xticks(range(e_.shape[1]), [f'f={v:g}' for v in e_.columns]); ax.set_yticks(range(e_.shape[0]), [f'a={v:g}' for v in e_.index])
for i in range(e_.shape[0]):
    for j in range(e_.shape[1]):
        ax.text(j, i, f'{e_.values[i, j]:.4f}', ha='center', va='center', fontsize=7)
ax.set_title('Ask-context: change in cost vs binary (budget 0.3; blue = Ask helps)'); ax.grid(False); plt.colorbar(im, ax=ax, fraction=.046)
ax = axes[1, 2]
if 'fresh_cohort_gate' in TABLES:
    for g in ['U', 'UDMS']:
        r = TABLES['fresh_cohort_gate']; r = r[(r.harm == 'fn') & (r.gate == g)].sort_values('alpha')
        ax.errorbar(r.alpha, r.risk, yerr=[r.risk-r.risk_lo, r.risk_hi-r.risk], fmt='o-', ms=4, lw=1.2, color=PAL[g], label=g)
    ax.plot([0, .045], [0, .045], 'k--', lw=1); ax.legend(frameon=False, fontsize=7)
    ax.set(xlabel='target alpha', ylabel='realised risk on 2,000 fresh patients', title='Untouched-cohort guarantee check')
else:
    cc_ = cov[cov.group == 'all'].reset_index(drop=True)
    ax.errorbar(range(len(cc_)), cc_.coverage, yerr=[cc_.coverage-cc_.lo, cc_.hi-cc_.coverage], fmt='o', ms=4, lw=1, color=PAL['U'])
    ax.axhline(1-PLAN['conformal_alpha'], ls='--', color='k', lw=1); ax.set_xticks(range(len(cc_)), cc_.label, rotation=90, fontsize=7)
    ax.set(ylabel='coverage (patient level)', title='Conformal coverage, out of sample')
fig.tight_layout(); fig.savefig(fig_dir/'v19_overview.png', dpi=200); fig.savefig(fig_dir/'v19_overview.pdf'); plt.show()


fig2, ax2 = plt.subplots(1, 3, figsize=(17, 4.8))
ax = ax2[0]
for pol, col, ls in [('binary_U', PAL['U'], '-'), ('binary_UDMS', PAL['UDMS'], '-'), ('binary_L2D', PAL['UD'], '-'), ('ask_both_UDMS', '#000000', '--'),
                     ('binary_entropy', PAL['entropy'], ':'), ('random', '#999999', ':'), ('oracle_binary', PAL['oracle'], '--')]:
    r = DCV[DCV.policy == pol]; ax.plot(r.price_of_workload, r.net_benefit, ls, color=col, lw=1.4, label=pol)
ax.set(xlabel='price of one unit of clinician workload', ylabel='net benefit (cost avoided - price x workload)', title='Decision curve'); ax.legend(fontsize=7, frameon=False)
ax = ax2[1]
b = PER1000[(PER1000.comparison == 'Full gate vs binary baseline') & (PER1000.harm == 'fn')].sort_values('alpha')
ax.errorbar(b.alpha, b.escalations_avoided_per_1000, yerr=[b.escalations_avoided_per_1000-b.lo, b.hi-b.escalations_avoided_per_1000], fmt='o-', color=PAL['UDMS'], lw=1.3, ms=4)
ax.axhline(0, color='k', lw=.8); ax.set(xlabel='target burden alpha', ylabel='escalations avoided per 1,000 patients', title='Full gate vs binary baseline, equal guaranteed risk')
ax = ax2[2]
if len(SEEDS) > 1:
    for c_, m_ in [('full_vs_binary', 'o'), ('full_vs_entropy', 's'), ('full_vs_l2d', '^')]:
        ax.plot(SEEDS.seed, SEEDS[c_], m_+'-', lw=1.2, label=c_)
    ax.axhline(0, color='k', lw=.8); ax.legend(fontsize=7, frameon=False)
ax.set(xlabel='fold-assignment seed', ylabel='change in referral rate at primary target', title='Stability across fold assignments')
fig2.tight_layout(); fig2.savefig(fig_dir/'v19_decision_and_stability.png', dpi=200); fig2.savefig(fig_dir/'v19_decision_and_stability.pdf'); plt.show()

# ---------------------------------------------------------------- automatic scorecard: every sentence is computed from the numbers
def count(df, col='verdict'):
    return {k: int(v) for k, v in df[col].value_counts().items()}
G = RESULTS['guarantee_internal']; S = []
S.append(('Finite-sample guarantee holds out of sample (cross-fit, primary scenario)',
          f"{G['realised_le_alpha']}/{G['configurations']} gate configurations kept realised risk <= alpha", G['realised_le_alpha'] >= 0.9*G['configurations']))
if 'fresh_confirmation' in RESULTS:
    fc = RESULTS['fresh_confirmation']
    S.append(('Guarantee confirmed on untouched fresh patients', f"{fc['target_met']}/{fc['configs']} full-gate targets met ({fc['patients']:,} patients)", fc['target_met'] >= 0.9*fc['configs']))
rq1 = RQ1[RQ1.contrast == 'Full gate vs binary baseline']
S.append(('RQ1: full 5-signal gate needs less workload than the binary baseline at equal guaranteed risk',
          f"Holm-adjusted verdicts across {len(rq1)} targets: {count(rq1, 'verdict_holm')}", bool((rq1.verdict_holm == 'better').sum() > len(rq1)/2)))
for name, r in RQ1[RQ1.contrast.isin(['D adds to U', 'M adds to U+D', 'S adds to U+D+M'])].groupby('contrast'):
    S.append((f'RQ1 component: {name}', f"Holm-adjusted verdicts: {count(r, 'verdict_holm')}", bool((r.verdict_holm == 'better').sum() > len(r)/2)))
a = RESULTS['ask_primary_budget']
for key, label in [('context', 'ASK-CONTEXT'), ('lateral', 'ASK-LATERAL'), ('both', 'both Ask actions')]:
    S.append((f'RQ2: {label} lowers cost at the primary budget',
              f"delta cost {a[key]['delta']:+.5f} [{a[key]['lo']:+.5f}, {a[key]['hi']:+.5f}]", a[key]['verdict'] == 'better'))
_g3c = G3[G3.variant == 'ask_context']
S.append(('RQ2: ASK-CONTEXT lowers workload at equal guaranteed risk', f"Holm-adjusted verdicts: {count(_g3c, 'verdict_holm')}",
          bool((_g3c.verdict_holm == 'better').sum() > len(_g3c)/2)))
bp = pd.DataFrame(RESULTS['bodhi_proxy'])
S.append(('Cost-aware Ask beats cost-unaware "ask whenever context is missing" at the same workload',
          '; '.join(f"{r.policy}: {r.delta_cost_vs_reference:+.5f} [{r.lo:+.5f},{r.hi:+.5f}]" for r in bp.itertuples()), bool((bp.verdict == 'better').any())))
S.append(('ASK-CONTEXT pays off somewhere in the assumed cost grid', f"{RESULTS['ask_break_even']['context']['ask_better']}/{RESULTS['ask_break_even']['context']['cells']} cells",
          RESULTS['ask_break_even']['context']['ask_better'] > 0))
S.append(('ASK-LATERAL pays off somewhere in the assumed cost grid', f"{RESULTS['ask_break_even']['lateral']['ask_better']}/{RESULTS['ask_break_even']['lateral']['cells']} cells",
          RESULTS['ask_break_even']['lateral']['ask_better'] > 0))
rob = ROB
S.append(('ASK-CONTEXT result holds in every deployment scheme (incl. the held-out structured-masking scheme)',
          '; '.join(f"{r.scheme}: {r.ask_context_verdict}" for r in rob.itertuples()), bool((rob.ask_context_verdict == 'better').all())))
if 'fresh_ask_context' in RESULTS:
    f_ = RESULTS['fresh_ask_context']
    S.append(('ASK-CONTEXT confirmed on untouched fresh patients', f"delta cost {f_['delta_cost']:+.5f} [{f_['lo']:+.5f}, {f_['hi']:+.5f}]; ask rate {f_['ask_rate']:.1%}", f_['verdict'] == 'better'))
# ---- declared primary endpoint (declared after the V15 results were seen: it is a fixed target for replication, not a blind pre-registration)
pe = PLAN['primary_endpoint']
_r = RQ1[(RQ1.harm == pe['harm']) & (RQ1.alpha == pe['alpha']) & (RQ1.contrast == pe['contrast'])].iloc[0]
S.append((f"PRIMARY ENDPOINT (pooled cross-fit): {pe['contrast']} at {pe['harm']} target {pe['alpha']}",
          f"referral {_r.delta_referral:+.4f} [{_r.lo:+.4f}, {_r.hi:+.4f}], Holm p={_r.p_holm:.3f}; = {-1000*_r.delta_referral:.1f} fewer escalations per 1,000 patients", bool(_r.verdict_holm == 'better')))
if 'fresh_cohort_contrast' in TABLES:
    _f = TABLES['fresh_cohort_contrast']; _f = _f[(_f.harm == pe['harm']) & (_f.alpha == pe['alpha'])].iloc[0]
    S.append(('PRIMARY ENDPOINT on untouched fresh patients', f"referral {_f.delta_referral_full_vs_U:+.4f} [{_f.lo:+.4f}, {_f.hi:+.4f}], Holm p={_f.p_holm:.3f}", bool(_f.p_holm < .05 and _f.delta_referral_full_vs_U < 0)))
_l2 = RQ1[RQ1.contrast == 'Full gate vs learning-to-defer surrogate baseline']
S.append(('Full gate vs a learning-to-defer surrogate-loss baseline (Madras/Mozannar-Sontag style)', f"Holm-adjusted verdicts: {count(_l2, 'verdict_holm')}", bool((_l2.verdict_holm == 'better').sum() > len(_l2)/2)))
if 'seed_stability' in RESULTS:
    _ss = RESULTS['seed_stability']
    S.append((f"Stable across {_ss['seeds']} independent fold assignments", f"full-vs-binary referral change at the primary target: mean {_ss['mean']['full_vs_binary']:+.4f}, sd {_ss['sd'].get('full_vs_binary', float('nan')):.4f}; same sign in all seeds: {_ss['same_sign_full_vs_binary']}",
              bool(_ss['same_sign_full_vs_binary'])))
if 'label_robustness' in RESULTS:
    _lb = RESULTS['label_robustness']
    S.append(('Guarantee survives a change of labeler (NegBio)', f"{_lb['recalibrated_met']}/{_lb['configs']} met after recalibration; {_lb['shifted_met']}/{_lb['configs']} met if the gate keeps its CheXpert calibration", _lb['recalibrated_met'] == _lb['configs']))
tsg = pd.DataFrame(RESULTS['ask_missing_then_gate']['guarantee'])
S.append(('Asking for the missing context FIRST, then gating, lowers workload at equal guaranteed risk',
          f"verdicts: {count(tsg)}; break-even cost of one context request = {tsg.break_even_request_workload.median():.3f} of a full review (median over targets)",
          bool((tsg.verdict == 'better').sum() > len(tsg)/2)))
if 'external_nih' in RESULTS:
    _e = RESULTS['external_nih']; _c = pd.DataFrame(_e['contrasts']); _cu = _c[_c.comparison == 'Full gate vs U']
    S.append((f"EXTERNAL dataset (NIH ChestX-ray14, {_e['patients']:,} patients, {len(_e['shared_labels'])} shared labels): guarantee holds",
              f"{_e['target_met_mimic_calibrated']}/{_e['configs']} full-gate targets met with MIMIC calibration unchanged; {_e['target_met_nih_recalibrated']}/{_e['configs']} after recalibrating only the threshold; image-only macro AUROC {_e['macro_auroc']:.3f}",
              _e['target_met_mimic_calibrated'] == _e['configs'] and _e['target_met_nih_recalibrated'] == _e['configs']))
    S.append(('EXTERNAL dataset: full gate needs fewer escalations than the uncertainty-only gate', f"Holm-adjusted verdicts: {count(_cu, 'verdict_holm')}", bool((_cu.verdict_holm == 'better').sum() > len(_cu)/2)))
if 'critical_findings' in RESULTS:
    _cf = RESULTS['critical_findings']; _t = pd.DataFrame(_cf['table'])
    _ratio = np.mean([(_t[f'risk_{l}']/_t[f'target_{l}']).mean() for l in PLAN['crit_labels']])
    S.append((f"Per-critical-finding risk guarantees ({', '.join(PLAN['crit_labels'])}) hold simultaneously",
              f"{_cf['union_all_targets_met']}/{_cf['union_configs']} union-gate and {_cf['shared_all_targets_met']}/{_cf['shared_configs']} shared-threshold configurations met all four targets; realised risk was on average {_ratio:.0%} of the target (conservative); escalation {_t.referral.min():.0%}-{_t.referral.max():.0%}",
              _cf['union_all_targets_met'] == _cf['union_configs'] and _cf['shared_all_targets_met'] == _cf['shared_configs']))
    _m = pd.DataFrame(_cf['method_comparison'])
    S.append(('A single shared threshold meets all per-finding targets with fewer escalations than a union of per-finding gates',
              '; '.join(f"target {r.target_fraction:g}: union minus shared {r.delta_referral:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}]" for r in _m.itertuples()), bool((_m.lo > 0).all())))
    _cv = pd.DataFrame(_cf['full_vs_U'])
    S.append(('Per-finding guarantees: full signal set needs fewer escalations than uncertainty alone', f"Holm-adjusted verdicts: {count(_cv, 'verdict_holm')}", bool((_cv.verdict_holm == 'better').sum() > len(_cv)/2)))
if 'ensemble_signal' in RESULTS:
    _es = pd.DataFrame(RESULTS['ensemble_signal']['contrasts'])
    for _name, _d in _es.groupby('contrast'):
        S.append((f'Ensemble-disagreement signal E: {_name}', f"Holm-adjusted verdicts: {count(_d, 'verdict_holm')}", bool((_d.verdict_holm == 'better').sum() > len(_d)/2)))
if 'label_agreement' in RESULTS:
    S.append(('Second labeler (NegBio) agrees closely with CheXpert on the evaluation patients',
              f"macro Cohen kappa {RESULTS['label_agreement']['macro_kappa']:.3f} on {RESULTS['label_agreement']['patients_matched']:,} patients (both rule-based labelers: NOT an independent reference standard)",
              RESULTS['label_agreement']['macro_kappa'] > .8))
if 'external_nih_subgroups' in RESULTS:
    _g = RESULTS['external_nih_subgroups']
    S.append(('EXTERNAL subgroup audit (NIH sex and age): risk targets hold in every subgroup',
              f"point estimate within target in {_g['point_target_met']}/{_g['rows']} subgroup configurations; upper 95% bound within target in {_g['upper_bound_met']}/{_g['rows']}; {_g['gaps_with_interval_excluding_zero']}/{_g['gaps']} subgroup risk gaps had an interval excluding zero",
              _g['point_target_met'] == _g['rows']))
if 'mimic_subgroups' in RESULTS:
    _m = RESULTS['mimic_subgroups']
    S.append((f"MIMIC subgroup audit ({', '.join(_m['groups'])}): risk targets hold in every subgroup",
              f"point estimate within target in {_m['point_target_met']}/{_m['configs']} subgroup configurations; upper 95% bound within target in {_m['upper_bound_met']}/{_m['configs']}; macro-AUROC range {_m['auroc_range'][0]:.3f}-{_m['auroc_range'][1]:.3f}",
              _m['point_target_met'] == _m['configs']))
sim = RESULTS['rq3_simulation']
S.append(('RQ3 (simulation only): workload-aware thresholds reduce harm at equal volume', f"{sim['aware_lower_harm_same_volume']}/{sim['settings']} simulated settings",
          sim['aware_lower_harm_same_volume'] > sim['settings']/2))
det = RESULTS['stress_mismatch']['detection_auroc']
S.append(('Signal 2 stress test: mismatched clinical context is detectable', f"detection AUROC {det['D signal (mean JS divergence)']:.3f} (0.5 = chance)", det['D signal (mean JS divergence)'] > .6))
s3 = SIG3[SIG3.scheme == 'struct40'].iloc[0]
S.append(('Signal 3 under withheld context: gate predicts when asking helps (held-out structured scheme)',
          f"corr(predicted gain, true gain) = {s3.predicted_gain_corr:.3f}", bool(s3.predicted_gain_corr > .2)))
S.append(('Signal 1: conformal sets reach nominal coverage', f"{RESULTS['conformal']['labels_all_group_upper_ci_below_nominal']} labels with upper CI below nominal",
          RESULTS['conformal']['labels_all_group_upper_ci_below_nominal'] <= 2))
SCORE = pd.DataFrame(S, columns=['claim', 'evidence', 'supported']); TABLES['scorecard'] = SCORE
pd.set_option('display.max_colwidth', 220); display(SCORE)

def to_py(o):
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, (float, np.floating)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o
RESULTS['scorecard'] = SCORE.to_dict('records')
json_write(OUT/'results_v19.json', to_py(RESULTS))
with pd.ExcelWriter(OUT/'master_v19.xlsx') as xw:
    for name, df in TABLES.items():
        df.astype({c: str for c in df.columns if df[c].dtype == object}).to_excel(xw, sheet_name=name[:31], index=False)
for name, df in TABLES.items():
    df.to_csv(OUT/f'{name}.csv', index=False)
lines = ['# Adaptive Risk Gate V19 - stable, Holm-corrected, guarantee-calibrated evaluation', '',
         f'{NE:,} patients evaluated out of sample (cached embeddings only). Wording below is generated from the numbers.', '',
         '| claim | evidence | supported |', '|---|---|---|']
lines += [f"| {r.claim} | {r.evidence} | {'yes' if r.supported else 'no'} |" for r in SCORE.itertuples()]
lines += ['', '## Standing limitations',
          '- Retrospective, single-source (MIMIC-CXR); labels are report-derived (CheXpert). No clinician outcomes.',
          '- ASK-CONTEXT is evaluated in a SIMULATED deployment in which the clinical history/indication is withheld at triage for some patients; the information it returns is the real cached context embedding.',
          '- Expert accuracy, request cost/failure and queue/override behaviour are assumptions (sensitivity grids provided), not measurements.',
          '- Whole-context absence is tested; partial section masking would need re-embedding of modified reports.',
          '- Fresh-cohort confirmation has no lateral view (ASK-LATERAL untested there).',
          '- The primary endpoint was declared after the V15 results were seen; confirmation needs new data (external dataset or new patients).',
          '- NegBio and CheXpert are both rule-based labelers with near-identical output (kappa about 0.98), so the label-source check is weak evidence of robustness; an independent reference (CheXbert or radiologist labels) was not available.',
          '- Subgroup (sex, age) audit on NIH is underpowered for rare findings; the MIMIC subgroup audit needs MIMIC-IV demographics and runs only if they are provided.',
          '- Per-critical-finding guarantees are conservative (few miss events per calibration fold); the ensemble signal was evaluated on one fold assignment only.',
          '- External validation covers 8 labels shared with NIH ChestX-ray14, image only (no clinical context, no lateral view); NIH labels differ in definition and are sparse.',
          '- Head regularisation strengths were explored on tuning patients of the pooled development data; only the fresh cohort is untouched by any design choice.']
(OUT/'Results_Summary_V19.md').write_text('\n'.join(lines), encoding='utf-8')
print('Saved to', OUT); print('\n'.join(lines[:24]))


## 10. Shareable results bundle
Exports aggregate tables only: no patient-level arrays and no fitted models.

In [ ]:
import zipfile
zip_path = OUT/'V19_RESULTS.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and 'PRIVATE' not in p.name and p.suffix in ('.csv', '.json', '.md', '.xlsx'):
            z.write(p, p.name)
    for p in sorted((OUT/'figures').glob('*')):
        z.write(p, 'figures/'+p.name)
print('Share-safe results bundle (no patient-level arrays):', zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
